# FULL PAPER RUN — everything, 10 seeds, results saved to file

This notebook runs **every experiment in the paper** and writes all numbers to
`results/paper_results.jsonl` + `results/paper_tables.txt`, which you then send
to build the manuscript tables.

**How to use**
1. Runtime → Change runtime type → **GPU** → Save
2. Runtime → **Run all**
3. When finished, run the LAST cell to print/download everything

⚠️ Full run with 10 seeds takes roughly **2–4 hours** on a GPU. If you are short
on time, change `SEEDS = 10` to `SEEDS = 5` in the config cell.

Each heavy cell is separate, so if the session drops you only re-run that cell.


## 0. Config — change SEEDS here


In [ ]:
SEEDS  = 10     # seeds for statistics (10 = paper quality, 5 = faster)
ROUNDS = 40     # federated rounds per run
print('SEEDS =', SEEDS, '| ROUNDS =', ROUNDS)


## 1. Setup — data, code, folders


In [ ]:
import os
os.makedirs('data', exist_ok=True); os.makedirs('results', exist_ok=True)
os.makedirs('fairaudit', exist_ok=True)
!rm -rf FedCrime && git clone --depth 1 -q https://github.com/vanetlabiitj/FedCrime.git
!cp FedCrime/Dataset/processed_crime.csv data/la_crime.csv
import torch, pandas as pd
print('GPU:', torch.cuda.is_available())
print('LA rows:', len(pd.read_csv('data/la_crime.csv')))


In [ ]:
%%writefile fairaudit/__init__.py
"""
fairaudit — detect base-rate confounding in group-fairness evaluation.
================================================================================
Group performance gaps computed with base-rate-DEPENDENT metrics (F1, precision,
accuracy, Precision@k, MAE) are non-zero even when two groups receive EQUAL
skill, because the attainable value of those metrics depends on the group's base
rate.  This package reports the confounded gap, the attainability floor, the
skill-normalised gap, and base-rate-INVARIANT gaps side by side.

Quick start
-----------
    import numpy as np
    from fairaudit import audit

    report = audit(y_true, y_score, groups, threshold=0.5)
    print(report)

`y_true`  (N,) or (N, C) binary labels
`y_score` same shape, model scores in [0, 1]
`groups`  (N,) group label per row (e.g. "Head"/"Tail")

Theory and proofs: see fairaudit.extensions and the accompanying paper.
Builds on Davis & Goadrich (2006), Boyd et al. (2012), Chouldechova (2017).
"""
from .metrics import (DEPENDENCE, auc, balanced_accuracy, accuracy, f1,
                      precision, tpr, fpr, attainable, attainable_f1, skill,
                      demographic_parity, equal_opportunity, equalized_odds,
                      predictive_parity)
from .audit import audit, AuditReport

__version__ = "0.1.0"
__all__ = ["audit", "AuditReport", "DEPENDENCE", "auc", "balanced_accuracy",
           "accuracy", "f1", "precision", "tpr", "fpr", "attainable",
           "attainable_f1", "skill", "demographic_parity", "equal_opportunity",
           "equalized_odds", "predictive_parity"]



In [ ]:
%%writefile fairaudit/metrics.py
"""
Fairness and performance metrics, classified by base-rate dependence.
================================================================================
Every metric below is annotated with whether its ATTAINABLE value depends on the
group's base rate p.  Metrics whose attainability depends on p produce non-zero
group gaps even under equal skill, and must not be used alone to claim
unfairness.

    CONFOUNDED  : F1, precision, accuracy, average precision,
                  demographic parity, predictive parity
    INVARIANT   : AUC-ROC, balanced accuracy, equal opportunity (TPR gap),
                  equalized odds (TPR+FPR gaps)
    CONDITIONAL : recall/TPR and FPR alone are attainability-invariant but
                  OPERATING-POINT dependent; report them as a pair, or fix the
                  operating point across groups.
"""
from __future__ import annotations

import numpy as np

DEPENDENCE = {
    "f1": "confounded",
    "precision": "confounded",
    "accuracy": "confounded",
    "average_precision": "confounded",
    "demographic_parity": "confounded",
    "predictive_parity": "confounded",
    "auc": "invariant",
    "balanced_accuracy": "invariant",
    "equal_opportunity": "invariant",
    "equalized_odds": "invariant",
    "tpr": "conditional",
    "fpr": "conditional",
}


def _conf(y, pred):
    y = np.asarray(y).ravel(); pred = np.asarray(pred).ravel()
    tp = float(np.sum((pred == 1) & (y == 1))); fp = float(np.sum((pred == 1) & (y == 0)))
    fn = float(np.sum((pred == 0) & (y == 1))); tn = float(np.sum((pred == 0) & (y == 0)))
    return tp, fp, fn, tn


def _safe(a, b):
    return 0.0 if b == 0 else a / b


# ------------------------------- performance ------------------------------- #
def f1(y, pred):
    tp, fp, fn, _ = _conf(y, pred); return _safe(2 * tp, 2 * tp + fp + fn)


def precision(y, pred):
    tp, fp, _, _ = _conf(y, pred); return _safe(tp, tp + fp)


def tpr(y, pred):
    tp, _, fn, _ = _conf(y, pred); return _safe(tp, tp + fn)


def fpr(y, pred):
    _, fp, _, tn = _conf(y, pred); return _safe(fp, fp + tn)


def accuracy(y, pred):
    tp, fp, fn, tn = _conf(y, pred); return _safe(tp + tn, tp + fp + fn + tn)


def balanced_accuracy(y, pred):
    return 0.5 * (tpr(y, pred) + (1.0 - fpr(y, pred)))


def auc(y, score):
    """Mann-Whitney AUC with tie correction; 0.5 = no skill, base-rate invariant."""
    y = np.asarray(y).ravel(); s = np.asarray(score, dtype=float).ravel()
    s = np.nan_to_num(s, nan=0.0, posinf=1.0, neginf=0.0)
    n_pos = int(y.sum()); n_neg = len(y) - n_pos
    if n_pos == 0 or n_neg == 0:
        return 0.5
    order = np.argsort(s, kind="mergesort")
    ranks = np.empty(len(s), float); ranks[order] = np.arange(1, len(s) + 1)
    s_sorted = s[order]
    _, first, counts = np.unique(s_sorted, return_index=True, return_counts=True)
    for st, ct in zip(first[counts > 1], counts[counts > 1]):
        idx = order[st:st + ct]; ranks[idx] = ranks[idx].mean()
    return float((ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


# --------------------------- group-fairness criteria ----------------------- #
def demographic_parity(pred_a, pred_b):
    """P(pred=1 | A) - P(pred=1 | B).  CONFOUNDED: depends on base rates."""
    return float(np.mean(pred_a) - np.mean(pred_b))


def equal_opportunity(y_a, p_a, y_b, p_b):
    """TPR_A - TPR_B.  INVARIANT: each TPR conditions on the positive class."""
    return tpr(y_a, p_a) - tpr(y_b, p_b)


def equalized_odds(y_a, p_a, y_b, p_b):
    """(TPR gap, FPR gap).  INVARIANT: both terms condition on a single class."""
    return (tpr(y_a, p_a) - tpr(y_b, p_b), fpr(y_a, p_a) - fpr(y_b, p_b))


def predictive_parity(y_a, p_a, y_b, p_b):
    """Precision gap.  CONFOUNDED: precision is bounded by the base rate.

    This is the criterion at the heart of Chouldechova (2017): it cannot hold
    simultaneously with equalized odds when base rates differ.
    """
    return precision(y_a, p_a) - precision(y_b, p_b)


# ------------------------------- attainability ----------------------------- #
def attainable_f1(p):
    """A_F1(p) = 2p/(1+p): F1 of the skill-free always-positive predictor."""
    p = np.asarray(p, dtype=float)
    return 2 * p / (1 + p)


def attainable(metric: str, p):
    """Skill-free attainable value of `metric` at base rate p."""
    p = np.asarray(p, dtype=float)
    if metric in ("f1",):
        return attainable_f1(p)
    if metric in ("precision", "average_precision"):
        return p                      # always-positive precision = p
    if metric == "accuracy":
        return np.maximum(p, 1 - p)   # majority-class rule
    if metric in ("auc", "balanced_accuracy"):
        return np.full_like(p, 0.5)   # invariant
    if metric in ("tpr",):
        return np.ones_like(p)
    raise ValueError(f"no attainability defined for {metric!r}")


def skill(metric: str, value, p):
    """RATIO form: value / A(p).  1.0 = no better than skill-free.

    CAUTION: unstable when A(p) is small (sparse groups), because a small
    absolute gain becomes a large ratio.  Prefer `skill_score` for group
    comparisons; this form is kept for interpretability ("% of attainable").
    """
    a = attainable(metric, p)
    return float(value) / float(np.asarray(a).mean() if np.ndim(a) else a)


def skill_score(metric: str, value, p, perfect: float = 1.0):
    """SKILL SCORE form:  (M - A) / (perfect - A).

    The standard forecasting-skill normalisation (cf. Brier Skill Score).
    0 = matches the skill-free baseline, 1 = perfect, <0 = worse than baseline.
    Unlike the ratio it stays bounded and comparable across groups with very
    different base rates, so it is the recommended statistic for group gaps.
    """
    a = attainable(metric, p)
    a = float(np.asarray(a).mean() if np.ndim(a) else a)
    denom = perfect - a
    return float("nan") if abs(denom) < 1e-12 else (float(value) - a) / denom



In [ ]:
%%writefile fairaudit/extensions.py
r"""
Extending the attainability decomposition beyond binary F1.
================================================================================
The core result (binary case) is

        Gap_M = [A_M(p_H) - A_M(p_T)]  +  skill term,     A_F1(p) = 2p/(1+p).

This module derives and NUMERICALLY VERIFIES the analogous confound for three
further settings that are common in spatio-temporal prediction.

--------------------------------------------------------------------------------
E1. MULTI-LABEL (macro-averaged) metrics
--------------------------------------------------------------------------------
With C labels of rates p_1..p_C, macro-F1 attainability is the MEAN of the
per-label ceilings:

        A_macro(p_1..p_C) = (1/C) sum_c 2 p_c / (1 + p_c).

IMPORTANT: this is NOT 2 pbar/(1 + pbar) for the pooled rate pbar. Because
x -> 2x/(1+x) is concave, Jensen gives

        (1/C) sum_c A(p_c)  <=  A(pbar),

so using the pooled rate OVERSTATES the ceiling. Papers reporting only pooled
sparsity therefore cannot recover the correct ceiling; per-label rates are
required.

--------------------------------------------------------------------------------
E2. RANKING metrics (Precision@k, NDCG@k)
--------------------------------------------------------------------------------
For a skill-free ranker over N items with positive rate p, the expected
Precision@k is p for every k. Hence

        A_P@k(p) = p,

which is linear (not concave) in p, and the group gap under equal skill is
exactly p_H - p_T. Ranking metrics are therefore ALSO confounded, and the
confound is larger than for F1 in the low-rate regime (since 2p/(1+p) > p).

--------------------------------------------------------------------------------
E3. REGRESSION metrics (MAE, RMSE)
--------------------------------------------------------------------------------
For counts with mean mu and variance sigma^2, a skill-free predictor emitting
the group mean attains

        A_MAE  = E|Y - mu|,      A_RMSE = sigma.

For Poisson counts sigma = sqrt(mu), so BOTH error metrics grow with the group
mean. Reporting raw MAE/RMSE gaps across groups with different intensities is
confounded in the OPPOSITE direction to F1: busier groups look WORSE.
This explains a pattern noted in the crime-prediction benchmark literature,
where models appear to degrade on high-volume regions under MAE/RMSE while
appearing to improve on them under F1.

Normalised alternatives (relative error MAE/mu, or a skill score
1 - MAE/A_MAE) remove the scale term.

Run:  python -m fairaudit.extensions
"""
from __future__ import annotations

import numpy as np

rng = np.random.default_rng(0)


# --------------------------------------------------------------------------- #
def a_macro_f1(rates):
    r = np.asarray(rates, dtype=float)
    return float(np.mean(2 * r / (1 + r)))


def a_pooled_f1(rates):
    pbar = float(np.mean(rates))
    return 2 * pbar / (1 + pbar)


def a_precision_at_k(p):
    return float(p)


def a_mae_poisson(mu, n=200_000):
    y = rng.poisson(mu, n)
    return float(np.mean(np.abs(y - mu)))


def a_rmse_poisson(mu):
    return float(np.sqrt(mu))


def hdr(t):
    print("\n" + "=" * 76); print(t); print("=" * 76)


def main():
    # ---------------------------------------------------------------- E1
    hdr("E1 — MULTI-LABEL: per-label mean vs pooled rate (Jensen gap)")
    cases = {
        "LA S-Omega (FedCrime Tab.1)": [1 - s / 100 for s in
                                        [37.21, 71.54, 72.50, 73.88,
                                         58.98, 65.43, 82.73, 88.16]],
        "CHI S-Alpha (FedCrime Tab.1)": [1 - s / 100 for s in
                                         [47.26, 5.83, 38.11, 38.50,
                                          29.94, 4.50, 17.60, 34.14]],
    }
    print(f"{'case':32s} {'correct A':>10s} {'pooled A':>10s} {'overstatement':>14s}")
    print("-" * 70)
    for name, rates in cases.items():
        a, ap = 100 * a_macro_f1(rates), 100 * a_pooled_f1(rates)
        print(f"{name:32s} {a:10.2f} {ap:10.2f} {ap - a:+14.2f}")
    print("\n  Using the pooled rate overstates the ceiling (Jensen). Papers that")
    print("  publish only aggregate sparsity cannot recover the correct ceiling.")

    # ---------------------------------------------------------------- E2
    hdr("E2 — RANKING: Precision@k is confounded, and worse than F1 at low p")
    print(f"{'p':>6} {'A_P@k = p':>11} {'A_F1 = 2p/(1+p)':>17} {'ratio':>8}")
    print("-" * 46)
    for p in [0.05, 0.10, 0.16, 0.30, 0.50]:
        af1 = 2 * p / (1 + p)
        print(f"{p:6.2f} {p:11.3f} {af1:17.3f} {af1 / p:8.2f}")
    print("\n  Verification (skill-free ranker, N=20000, k=100, 200 trials):")
    for p in [0.05, 0.20, 0.50]:
        vals = []
        for _ in range(200):                        # average away sampling noise
            y = (rng.random(20000) < p).astype(int)
            s = rng.random(20000)                   # scores independent of y
            vals.append(y[np.argsort(-s)[:100]].mean())
        print(f"    p={p:.2f}  empirical P@100 = {np.mean(vals):.3f} "
              f"+/- {np.std(vals):.3f}  (theory {p:.3f})")

    # ---------------------------------------------------------------- E3
    hdr("E3 — REGRESSION: MAE/RMSE grow with the group mean (opposite direction)")
    print(f"{'mu':>6} {'A_MAE':>9} {'A_RMSE':>9} {'relative MAE (A/mu)':>21}")
    print("-" * 48)
    for mu in [0.5, 1.0, 2.0, 5.0, 10.0]:
        am, ar = a_mae_poisson(mu), a_rmse_poisson(mu)
        print(f"{mu:6.1f} {am:9.3f} {ar:9.3f} {am / mu:21.3f}")
    print("\n  A skill-free predictor's error RISES with intensity, so busy regions")
    print("  look worse under raw MAE/RMSE while looking better under F1. Both are")
    print("  attainability effects, not skill differences.")

    hdr("SUMMARY — attainability by metric family")
    print("  binary F1        A(p) = 2p/(1+p)          confounded (concave)")
    print("  macro-F1         mean_c 2p_c/(1+p_c)      confounded; needs per-label p")
    print("  precision        A(p) = p                 confounded (linear)")
    print("  Precision@k      A(p) = p                 confounded (linear)")
    print("  MAE (Poisson)    A(mu) = E|Y-mu|          confounded, increasing in mu")
    print("  RMSE (Poisson)   A(mu) = sqrt(mu)         confounded, increasing in mu")
    print("  AUC / bal. acc.  A = 0.5                  INVARIANT")


if __name__ == "__main__":
    main()



In [ ]:
%%writefile fairaudit/audit.py
"""The main audit entry point: one call, a full confounding report."""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List

import numpy as np

from . import metrics as M


@dataclass
class AuditReport:
    groups: List[str]
    base_rate: Dict[str, float] = field(default_factory=dict)
    confounded: Dict[str, Dict[str, float]] = field(default_factory=dict)
    invariant: Dict[str, Dict[str, float]] = field(default_factory=dict)
    attainable: Dict[str, float] = field(default_factory=dict)
    skill: Dict[str, float] = field(default_factory=dict)
    skill_score: Dict[str, float] = field(default_factory=dict)
    gaps: Dict[str, float] = field(default_factory=dict)
    verdict: str = ""

    def __str__(self) -> str:
        hi, lo = self.groups[0], self.groups[-1]
        L = ["=" * 74,
             f"FAIRNESS AUDIT — base-rate confounding check ({hi} vs {lo})",
             "=" * 74,
             f"{'group':10s} {'base rate':>10s} {'A_F1(p)':>9s} {'F1':>8s} "
             f"{'%attain':>8s} {'skillsc':>8s} {'AUC':>8s}",
             "-" * 70]
        for g in self.groups:
            L.append(f"{g:10s} {self.base_rate[g]:10.3f} "
                     f"{100*self.attainable[g]:9.1f} "
                     f"{100*self.confounded['f1'][g]:8.2f} "
                     f"{100*self.skill[g]:8.1f} "
                     f"{100*self.skill_score[g]:8.2f} "
                     f"{100*self.invariant['auc'][g]:8.2f}")
        L += ["", "GAPS (first group minus last)",
              f"{'metric':22s} {'gap':>9s}   class"]
        L.append("-" * 46)
        for k, v in self.gaps.items():
            L.append(f"{k:22s} {100*v:9.2f}   {M.DEPENDENCE.get(k, '-')}")
        L += ["", self.verdict, "=" * 74]
        return "\n".join(L)


def _as2d(a):
    a = np.asarray(a)
    return a.reshape(-1, 1) if a.ndim == 1 else a.reshape(len(a), -1)


def audit(y_true, y_score, groups, threshold=0.5, per_label_threshold=None
          ) -> AuditReport:
    """Audit group gaps for base-rate confounding.

    Parameters
    ----------
    y_true  : (N,) or (N, C) binary labels
    y_score : same shape, scores in [0, 1]
    groups  : (N,) group label per row
    threshold : scalar decision threshold (ignored where per_label_threshold set)
    per_label_threshold : optional (C,) per-label thresholds

    Returns
    -------
    AuditReport with base rates, attainability, confounded and invariant gaps.
    """
    Y, S = _as2d(y_true), _as2d(y_score)
    S = np.nan_to_num(S, nan=0.0, posinf=1.0, neginf=0.0)
    g = np.asarray(groups).ravel()
    if len(Y) != len(g):
        raise ValueError("y_true and groups must have the same length")
    C = Y.shape[1]
    thr = (np.full(C, threshold) if per_label_threshold is None
           else np.asarray(per_label_threshold, dtype=float))
    P = (S > thr.reshape(1, -1)).astype(int)

    # order groups by base rate, highest first (Head -> Tail convention)
    uniq = list(dict.fromkeys(g.tolist()))
    uniq.sort(key=lambda u: -float(Y[g == u].mean()))

    rep = AuditReport(groups=uniq)
    rep.confounded = {k: {} for k in ("f1", "precision", "accuracy")}
    rep.invariant = {k: {} for k in ("auc", "balanced_accuracy")}

    for u in uniq:
        m = g == u
        yg, pg, sg = Y[m], P[m], S[m]
        rates = yg.mean(axis=0)
        rep.base_rate[u] = float(rates.mean())
        rep.attainable[u] = float(np.mean(M.attainable_f1(rates)))  # macro (E1)

        def per_label(fn, use_score=False):
            vals = []
            for c in range(C):
                if yg[:, c].sum() in (0, len(yg)):
                    continue
                vals.append(fn(yg[:, c], sg[:, c] if use_score else pg[:, c]))
            return float(np.mean(vals)) if vals else float("nan")

        rep.confounded["f1"][u] = per_label(M.f1)
        rep.confounded["precision"][u] = per_label(M.precision)
        rep.confounded["accuracy"][u] = per_label(M.accuracy)
        rep.invariant["auc"][u] = per_label(M.auc, use_score=True)
        rep.invariant["balanced_accuracy"][u] = per_label(M.balanced_accuracy)
        rep.skill[u] = rep.confounded["f1"][u] / max(rep.attainable[u], 1e-9)
        rep.skill_score[u] = M.skill_score("f1", rep.confounded["f1"][u], rates)

    hi, lo = uniq[0], uniq[-1]
    for k in ("f1", "precision", "accuracy"):
        rep.gaps[k] = rep.confounded[k][hi] - rep.confounded[k][lo]
    for k in ("auc", "balanced_accuracy"):
        rep.gaps[k] = rep.invariant[k][hi] - rep.invariant[k][lo]
    rep.gaps["skill_ratio_f1"] = rep.skill[hi] - rep.skill[lo]
    rep.gaps["skill_score_f1"] = rep.skill_score[hi] - rep.skill_score[lo]

    # verdict
    gf1, gauc = abs(rep.gaps["f1"]), abs(rep.gaps["auc"])
    floor = float(rep.attainable[hi] - rep.attainable[lo])
    ratio = gf1 / max(gauc, 1e-9)
    if gf1 > 0.05 and ratio > 3:
        rep.verdict = (
            f"VERDICT: likely CONFOUNDED. The F1 gap ({100*gf1:.1f} pts) is "
            f"{ratio:.0f}x the AUC gap ({100*gauc:.1f} pts),\n"
            f"and the attainability floor alone predicts {100*floor:.1f} pts. "
            "Report an invariant\nmetric or the skill-normalised gap before "
            "claiming a disparity in model quality.")
    else:
        rep.verdict = (
            "VERDICT: no strong evidence of base-rate confounding; the "
            "confounded and\ninvariant metrics broadly agree.")
    return rep



In [ ]:
%%writefile robust_fair_gnn.py
"""
Robust & Fair Federated GNN for Crime Prediction under Extreme Sparsity
================================================================================
Research extension of FedCrime. Runs on the REAL Los Angeles / Chicago data.

It brings together, in ONE framework:
  * ZINB zero-inflation loss                       (from the FedCrime paper)
  * A Spatio-Temporal GNN (TCN + graph convolution) over a learned region graph
  * ATTACKS by malicious clients, incl. the novel *sparsity-camouflaged* attack
  * DEFENSES, incl. a novel density/graph "vouching" aggregator
  * FAIRNESS metrics (Head / Mid / Tail performance gap)
  * WEEK-AHEAD prediction (--horizon) and UNCERTAINTY (MC-dropout)

The research question (the gap): in federated crime prediction, honest SPARSE
neighbourhoods look like MALICIOUS clients, so standard defenses either let
attacks through or unfairly silence poor regions. We study this
sparsity-robustness-fairness trilemma and a defense that resolves it.

Examples
--------
# clean run on LA, next-day prediction, plain FedAvg:
python3 robust_fair_gnn.py --city la

# compare defenses under the sparsity-camouflaged attack (the key experiment):
python3 robust_fair_gnn.py --city la --attack camouflage --compare

# week-ahead (7 days) with uncertainty:
python3 robust_fair_gnn.py --city la --horizon 7 --mc 10

Requires: torch, numpy, pandas, scikit-learn
================================================================================
"""
import argparse
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score

EPS = 1e-10
C = 8                       # crime categories
L = 8                       # input days per window

CITY = {
    "la":      ("data/la_crime.csv",  "date_occ", "2018-01-01", "2018-12-31"),
    "chicago": ("data/chi_crime.csv", "date_occ", "2015-01-01", "2015-12-31"),
    "nyc":     ("data/nyc_crime.csv", "date_occ", "2019-01-01", "2019-12-31"),
    "sf":      ("data/sf_crime.csv",  "date_occ", "2019-01-01", "2019-12-31"),
}


# --------------------------------------------------------------------------- #
# 1. REAL DATA -> aligned (days x regions x categories) tensor + region graph
# --------------------------------------------------------------------------- #
def load_city(csv, date_col, start, end):
    df = pd.read_csv(csv)
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col])
    days = pd.date_range(start, end, freq="D")
    regions = sorted(df["neighborhood_id"].unique())
    ridx = {r: i for i, r in enumerate(regions)}
    didx = {d: i for i, d in enumerate(days)}
    mat = np.zeros((len(days), len(regions), C), dtype=np.float32)
    g = df.groupby([df[date_col].dt.normalize(), "neighborhood_id", "crime_type_id"]).size()
    for (day, r, c), _ in g.items():
        if day in didx and r in ridx and 0 <= int(c) < C:
            mat[didx[day], ridx[r], int(c)] = 1.0
    return mat, regions                        # mat: (D, R, C) binary


def make_windows(mat, horizon=1, dow=True):
    """X: (N, L, R, C[+2]), Y: (N, R, C) predicting `horizon` steps ahead.

    dow=True appends two day-of-week signals (sin/cos of the TARGET day) to every
    region's channel vector. Crime has strong weekly rhythms, so this gives the
    model genuine predictive signal beyond the trivial base rate.
    """
    D = mat.shape[0]
    X, Y = [], []
    for t in range(D - L - horizon + 1):
        win = mat[t:t + L]                       # (L, R, C)
        if dow:
            tgt = t + L + horizon - 1            # index of the day we predict
            ang = 2 * np.pi * (tgt % 7) / 7.0
            extra = np.zeros((L, win.shape[1], 2), np.float32)
            extra[:, :, 0] = np.sin(ang); extra[:, :, 1] = np.cos(ang)
            win = np.concatenate([win, extra], axis=2)   # (L, R, C+2)
        X.append(win)
        Y.append(mat[t + L + horizon - 1])
    return np.asarray(X, np.float32), np.asarray(Y, np.float32)


def build_graph(mat_train, k=6):
    """Region graph from crime-pattern similarity on the TRAINING period."""
    totals = mat_train.sum(axis=2)             # (Dtrain, R) daily total per region
    R = totals.shape[1]
    corr = np.corrcoef(totals.T)               # (R, R)
    corr = np.nan_to_num(corr)
    np.fill_diagonal(corr, -1)
    A = np.eye(R, dtype=np.float32)
    for i in range(R):                         # connect each region to top-k similar
        for j in np.argsort(corr[i])[::-1][:k]:
            A[i, j] = 1.0
    A = np.maximum(A, A.T)                      # symmetric
    d = A.sum(1); dinv = 1.0 / np.sqrt(np.maximum(d, 1e-6))
    return (A * dinv[:, None] * dinv[None, :]).astype(np.float32), A


def head_mid_tail(mat_train):
    """Label each region Head/Mid/Tail by total training crime (20/30/50%)."""
    totals = mat_train.sum(axis=(0, 2))        # (R,)
    order = np.argsort(totals)[::-1]
    R = len(totals); nh = max(1, round(R * 0.2)); nm = max(1, round(R * 0.3))
    tag = np.empty(R, dtype=object)
    tag[order[:nh]] = "Head"; tag[order[nh:nh + nm]] = "Mid"; tag[order[nh + nm:]] = "Tail"
    return tag


# --------------------------------------------------------------------------- #
# 2. MODEL: ST-GNN (TCN temporal + graph conv) + ZINB + classifier + dropout
# --------------------------------------------------------------------------- #
class TCNBlock(nn.Module):
    def __init__(self, i, o, k=3, dil=1):
        super().__init__()
        p = (k - 1) * dil // 2
        self.c1 = nn.Conv1d(i, o, k, padding=p, dilation=dil)
        self.c2 = nn.Conv1d(o, o, k, padding=p, dilation=dil)
        self.r = nn.ReLU()
        self.res = nn.Conv1d(i, o, 1) if i != o else nn.Identity()

    def forward(self, x):
        s = self.res(x); x = self.r(self.c1(x)); x = self.r(self.c2(x)); return x + s


class STGNN(nn.Module):
    """gnn_type: 'plain' (basic GCN), 'gated' (fixes over-smoothing), 'attention'."""
    def __init__(self, hidden=16, use_graph=True, gnn_type="plain", p_drop=0.2,
                 in_ch=None):
        super().__init__()
        self.use_graph = use_graph
        self.gnn_type = gnn_type
        self.in_ch = in_ch if in_ch is not None else C
        self.tcn = nn.Sequential(TCNBlock(self.in_ch, hidden, dil=1),
                                 TCNBlock(hidden, hidden, dil=2),
                                 TCNBlock(hidden, hidden, dil=4))
        self.g1 = nn.Linear(hidden, hidden); self.g2 = nn.Linear(hidden, hidden)
        self.gate1 = nn.Linear(hidden, hidden); self.gate2 = nn.Linear(hidden, hidden)
        self.relu = nn.ReLU(); self.drop = nn.Dropout(p_drop)
        self.fc_pi = nn.Linear(hidden, C)      # Eq 6
        self.fc_mu = nn.Linear(hidden, C)      # Eq 7
        self.fc_phi = nn.Linear(hidden, C)     # Eq 8
        self.fc_out = nn.Linear(hidden, C)     # crime-present classifier

    def _gated(self, h, A, lin, gate):
        # gated residual: g=sigmoid(gate(h)); H = g*(A.H.W) + (1-g)*H
        # each region learns how much to trust neighbours vs its own signal
        agg = torch.einsum('ij,bjh->bih', A, lin(h))
        g = torch.sigmoid(gate(h))
        return self.relu(g * agg + (1 - g) * h)

    def _attn(self, h, A, lin):
        # attention over graph neighbours (masked by the adjacency)
        Wh = lin(h); Hd = Wh.shape[-1]
        scores = torch.einsum('bih,bjh->bij', Wh, Wh) / (Hd ** 0.5)
        mask = (A > 0).float().unsqueeze(0)
        scores = scores.masked_fill(mask == 0, float('-inf'))
        alpha = torch.softmax(scores, dim=-1)
        return self.relu(torch.einsum('bij,bjh->bih', alpha, Wh))

    def _mix(self, h, A):
        if not self.use_graph:                       # no graph (TCN only)
            return self.relu(self.g2(self.relu(self.g1(h))))
        if self.gnn_type == "gated":
            h = self._gated(h, A, self.g1, self.gate1)
            return self._gated(h, A, self.g2, self.gate2)
        if self.gnn_type == "attention":
            h = self._attn(h, A, self.g1)
            return self._attn(h, A, self.g2)
        # plain GCN
        h = self.relu(torch.einsum('ij,bjh->bih', A, self.g1(h)))
        return self.relu(torch.einsum('ij,bjh->bih', A, self.g2(h)))

    def forward(self, X, A):
        B, Lw, R, Ch = X.shape
        h = X.permute(0, 2, 3, 1).reshape(B * R, Ch, Lw)
        h = self.tcn(h)[:, :, -1].reshape(B, R, -1)
        h = self.drop(self._mix(h, A))
        pi = torch.sigmoid(self.fc_pi(h))
        mu = torch.exp(torch.clamp(self.fc_mu(h), max=15.0))
        phi = torch.nn.functional.softplus(self.fc_phi(h))
        return pi, mu, phi, self.fc_out(h)


def zinb_elem(pi, mu, phi, y):
    """Per-element ZINB negative log-likelihood, shape (B, R, C)."""
    is0 = y.eq(0).float(); is1 = y.gt(0).float()
    zero = is0 * torch.log(pi + EPS)
    nb = is1 * (torch.lgamma(y + phi) - torch.lgamma(phi) - torch.lgamma(y + 1.0)
                + phi * (torch.log(phi + EPS) - torch.log(phi + mu + EPS))
                + y * (torch.log(mu + EPS) - torch.log(phi + mu + EPS)))
    return -(zero + nb)


def zinb_loss(pi, mu, phi, y):
    return zinb_elem(pi, mu, phi, y).mean()


def loss_fn(pi, mu, phi, logit, y, pos_weight=None, region_weight=None,
            group_ids=None, dro_tau=0.0, lam=0.3):
    # pos_weight counters class imbalance. Group-DRO (dro_tau>0) adaptively
    # up-weights the WORST-performing group (the poor tail regions) each step,
    # so training focuses on closing the fairness gap instead of ignoring the
    # tail. region_weight is the older static fairness weighting.
    bce = torch.nn.functional.binary_cross_entropy_with_logits(
        logit, y, pos_weight=pos_weight, reduction="none")     # (B, R, C)
    loss = bce + lam * zinb_elem(pi, mu, phi, y)               # (B, R, C)
    if group_ids is not None and dro_tau > 0:
        lr = loss.mean(dim=(0, 2))                             # per-region loss (R,)
        groups = torch.unique(group_ids)
        gloss = torch.stack([lr[group_ids == g].mean() for g in groups])
        gw = torch.softmax(dro_tau * gloss, dim=0) * len(groups)   # worse -> higher
        rw = torch.ones_like(lr)
        for i, g in enumerate(groups):
            rw[group_ids == g] = gw[i]
        loss = loss * rw.view(1, -1, 1)
    elif region_weight is not None:
        loss = loss * region_weight.view(1, -1, 1)
    return loss.mean()


# --------------------------------------------------------------------------- #
# 3. ATTACKS (applied inside a malicious client)
# --------------------------------------------------------------------------- #
def poison_labels(Y, attack):
    if attack == "labelflip":
        return 1.0 - Y                          # flip yes<->no
    if attack == "camouflage":
        return np.zeros_like(Y)                 # pretend "no crime anywhere" (looks sparse)
    return Y


def poison_update(delta, attack, factor=8.0):
    if attack == "scale":
        return [d * factor for d in delta]      # blow up the update
    if attack == "camouflage":
        return [d * 0.3 for d in delta]         # small, sparse-looking update
    return delta


# --------------------------------------------------------------------------- #
# 4. FEDERATED TRAINING with pluggable DEFENSE
# --------------------------------------------------------------------------- #
def gw(net): return [p.detach().clone() for p in net.state_dict().values()]
def sw(net, w):
    sd = net.state_dict()
    for k, v in zip(sd.keys(), w): sd[k] = v.clone()
    net.load_state_dict(sd)
def flat(w): return torch.cat([t.flatten() for t in w])


def aggregate(defense, wsets, gwt, densities):
    """Combine client weight-sets into a new global model."""
    n = len(wsets)
    if defense == "fedavg":
        return [torch.stack([w[i] for w in wsets]).mean(0) for i in range(len(wsets[0]))]

    deltas = [[w[i] - gwt[i] for i in range(len(gwt))] for w in wsets]

    if defense == "strust2":
        # STRUST-v2: norm-clip to the median norm (kills scaling attacks
        # deterministically), then trust-weight by direction agreement with a
        # coordinate-wise MEDIAN reference (robust to a minority of attackers),
        # then aggregate. More stable than v1 across attacks/cities.
        flatd = [flat(d) for d in deltas]
        norms = torch.tensor([float(f.norm()) + 1e-9 for f in flatd])
        med = float(norms.median())
        clip = torch.clamp(med / norms, max=1.0)               # scale down only
        stacked = torch.stack([flatd[i] * clip[i] for i in range(n)])
        ref = stacked.median(0).values                          # robust reference
        ref = ref / (ref.norm() + 1e-9)
        cos = torch.stack([(stacked[i] / (stacked[i].norm() + 1e-9)) @ ref
                           for i in range(n)])
        trust = torch.clamp(cos, min=0.0) ** 2                  # sharpen
        if float(trust.sum()) < 1e-6:
            trust = torch.ones(n)
        trust = trust / trust.sum()
        out = []
        for li in range(len(gwt)):
            acc = sum(trust[i] * deltas[i][li] * clip[i] for i in range(n))
            out.append(gwt[li] + acc)
        return out

    if defense == "strust":
        # SPARSITY-AWARE TRUST  (our trilemma solution).
        # Standard defenses reject "outlier-distant" clients -> they wrongly
        # reject honest sparse (tail) regions. Instead we judge clients by the
        # DIRECTION of their update and neutralise magnitude:
        #   1) unit-normalise each update  -> a scaling attack becomes harmless
        #   2) trust = max(0, cos(update, robust median direction))
        #        -> label-flip / camouflage point the wrong way  -> trust 0
        #        -> honest sparse clients point the right way     -> trust kept
        #   3) trust-weighted average, rescaled to the typical honest magnitude.
        flatd = [flat(d) for d in deltas]
        norms = [float(f.norm()) + 1e-9 for f in flatd]
        units = torch.stack([flatd[i] / norms[i] for i in range(n)])
        ref = units.median(0).values
        ref = ref / (ref.norm() + 1e-9)
        trust = torch.clamp(units @ ref, min=0.0)              # direction agreement
        if float(trust.sum()) < 1e-6:
            trust = torch.ones(n)
        trust = trust / trust.sum()
        scale = float(torch.tensor(norms).median())
        out = []
        for li in range(len(gwt)):
            acc = sum(trust[i] * (deltas[i][li] / norms[i]) for i in range(n))
            out.append(gwt[li] + scale * acc)
        return out

    F = torch.stack([flat(d) for d in deltas])                 # (n, P)
    dist = torch.cdist(F, F)                                    # pairwise distances

    if defense == "trimmed":
        keep = max(1, n - 2)
        idx = torch.argsort(dist.sum(1))[:keep]
    elif defense == "krum":
        kk = max(1, n - 2)
        scores = torch.stack([torch.sort(dist[i])[0][1:kk + 1].sum() for i in range(n)])
        idx = torch.argsort(scores)[:max(1, n - 2)]
    elif defense == "vouch":
        # NOVEL: a client's "oddness" is EXPECTED to be high if it is sparse
        # (honest tail region). Divide the Krum score by the client's data
        # density so sparse-honest clients are NOT penalised, while dense-yet-
        # odd clients (real attackers) are. Then keep the low-suspicion clients.
        kk = max(1, n - 2)
        raw = torch.stack([torch.sort(dist[i])[0][1:kk + 1].sum() for i in range(n)])
        dens = torch.tensor(densities, dtype=raw.dtype).clamp(min=1e-3)
        suspicion = raw * dens                                 # density-adjusted
        idx = torch.argsort(suspicion)[:max(1, n - 2)]
    else:
        raise ValueError(defense)

    sel = [wsets[i] for i in idx.tolist()]
    return [torch.stack([w[i] for w in sel]).mean(0) for i in range(len(sel[0]))]


def train(clients, A_full, defense, attack, mal_ids, use_graph=True,
          rounds=60, local_epochs=5, lr=0.03, seed=0, pos_weight=None,
          gnn_type="plain", fair=False, dro_tau=0.0, in_ch=None):
    torch.manual_seed(seed)
    g = STGNN(use_graph=use_graph, gnn_type=gnn_type, in_ch=in_ch); gwt = gw(g)
    for _ in range(rounds):
        wsets, dens = [], []
        for ci, cl in enumerate(clients):
            local = STGNN(use_graph=use_graph, gnn_type=gnn_type, in_ch=in_ch); sw(local, gwt)
            A = torch.tensor(cl["A"]); X = torch.tensor(cl["X"])
            Y = torch.tensor(poison_labels(cl["Y"], attack if ci in mal_ids else "none"))
            opt = torch.optim.Adam(local.parameters(), lr=lr, weight_decay=1e-4)
            rw = cl.get("rw") if fair else None
            grp = cl.get("grp")
            local.train()
            for _ in range(local_epochs):
                opt.zero_grad()
                pi, mu, phi, logit = local(X, A)
                loss_fn(pi, mu, phi, logit, Y, pos_weight=pos_weight,
                        region_weight=rw, group_ids=grp,
                        dro_tau=dro_tau).backward(); opt.step()
            delta = [p - q for p, q in zip(gw(local), gwt)]
            if ci in mal_ids:
                delta = poison_update(delta, attack)
            wsets.append([q + d for q, d in zip(gwt, delta)])
            dens.append(float(cl["Y"].mean()))                 # client data density
        gwt = aggregate(defense, wsets, gwt, dens)
        sw(g, gwt)
    return g


# --------------------------------------------------------------------------- #
# 5. EVALUATION: overall + fairness (Head/Mid/Tail) + uncertainty
# --------------------------------------------------------------------------- #
@torch.no_grad()
def tune_thresholds(net, X, Y, A):
    """Per-(region, crime-type) decision threshold tuned on a VALIDATION set to
    maximise each cell's F1. Rare crimes in poor regions get a LOWER threshold
    so they are actually predicted -> higher tail F1 -> smaller fairness gap.
    This is the fairness fix: it targets the decision, not the loss."""
    net.eval()
    prob = torch.sigmoid(net(torch.tensor(X), torch.tensor(A))[3]).numpy()  # (N,R,C)
    R = Y.shape[1]
    grid = np.arange(0.03, 0.60, 0.02)
    thr = np.full((R, C), 0.5, dtype=np.float32)
    for r in range(R):
        for cc in range(C):
            y = Y[:, r, cc]
            if y.sum() == 0:                     # no positives to tune on
                continue
            p = prob[:, r, cc]
            best_t, best_f = 0.5, -1.0
            for t in grid:
                f = f1_score(y, (p > t).astype(float), zero_division=0)
                if f > best_f:
                    best_f, best_t = f, t
            thr[r, cc] = best_t
    return thr


@torch.no_grad()
def evaluate(net, X, Y, A, tags, mc=0, thr=None):
    net.eval()
    Xt = torch.tensor(X); At = torch.tensor(A)
    if mc > 0:                                   # MC-dropout uncertainty
        net.train()                              # keep dropout ON
        probs = torch.stack([torch.sigmoid(net(Xt, At)[3]) for _ in range(mc)])
        prob = probs.mean(0); uncertainty = probs.std(0).mean().item()
        net.eval()
    else:
        prob = torch.sigmoid(net(Xt, At)[3]); uncertainty = float("nan")
    prob = torch.nan_to_num(prob, nan=0.0, posinf=1.0, neginf=0.0)
    if thr is not None:                          # adaptive threshold
        tt = torch.tensor(thr, dtype=prob.dtype)
        th = tt.view(1, -1, 1) if tt.dim() == 1 else tt.unsqueeze(0)  # (1,R[,C])
        pred = (prob > th).float().numpy()
    else:
        pred = (prob > 0.5).float().numpy()      # (N, R, C)
    N, R, _ = pred.shape

    def group_f1(mask):
        yy = Y[:, mask, :].reshape(-1, C); pp = pred[:, mask, :].reshape(-1, C)
        return f1_score(yy, pp, average="macro", zero_division=0) * 100

    def group_ceiling(mask):
        """Max attainable macro-F1 for this group's base rates.

        For a label with positive rate p, precision <= p and recall <= 1, so
        F1 <= 2p/(1+p). This ceiling SCALES WITH BASE RATE, which is why sparse
        (tail) regions can never reach head-level raw F1 however good the model.
        """
        yg = Y[:, mask, :]
        p = yg.mean(axis=(0, 1))                       # per-category base rate
        return float((2 * p / (1 + p)).mean()) * 100

    overall = f1_score(Y.reshape(-1, C), pred.reshape(-1, C), average="macro", zero_division=0) * 100
    res = {"overall": overall, "uncertainty": uncertainty}

    # ---- PER-GROUP metric decomposition (tests the theory) ---------------- #
    # Base-rate DEPENDENT metrics (F1, precision, accuracy) should show a large
    # Head-Tail gap; base-rate INVARIANT metrics (AUC, balanced accuracy, TPR)
    # should show almost none -- on the SAME predictions.
    probn = np.nan_to_num(prob.numpy(), nan=0.0, posinf=1.0, neginf=0.0)
    for grp in ["Head", "Mid", "Tail"]:
        m = np.array([t == grp for t in tags])
        yg = Y[:, m, :].reshape(-1, C)
        pg = pred[:, m, :].reshape(-1, C)
        sg = probn[:, m, :].reshape(-1, C)
        f1s, precs, accs, aucs_g, bals, tprs = [], [], [], [], [], []
        for cc_ in range(C):
            y, p_, s_ = yg[:, cc_], pg[:, cc_], sg[:, cc_]
            if y.sum() == 0 or y.sum() == len(y):
                continue
            tp = ((p_ == 1) & (y == 1)).sum(); fp = ((p_ == 1) & (y == 0)).sum()
            fn = ((p_ == 0) & (y == 1)).sum(); tn = ((p_ == 0) & (y == 0)).sum()
            f1s.append(0.0 if (2*tp+fp+fn) == 0 else 2*tp/(2*tp+fp+fn))
            precs.append(0.0 if (tp+fp) == 0 else tp/(tp+fp))
            accs.append((tp+tn)/len(y))
            tpr = 0.0 if (tp+fn) == 0 else tp/(tp+fn)
            tnr = 0.0 if (tn+fp) == 0 else tn/(tn+fp)
            tprs.append(tpr); bals.append(0.5*(tpr+tnr))
            aucs_g.append(0.5 if np.allclose(s_, s_[0]) else roc_auc_score(y, s_))
        mean = lambda v: float(np.mean(v))*100 if v else float("nan")
        res[grp+"_f1"] = mean(f1s); res[grp+"_prec"] = mean(precs)
        res[grp+"_acc"] = mean(accs); res[grp+"_auc"] = mean(aucs_g)
        res[grp+"_bal"] = mean(bals); res[grp+"_tpr"] = mean(tprs)
    for k in ["f1", "prec", "acc", "auc", "bal", "tpr"]:
        res["gap_"+k] = res.get("Head_"+k, float("nan")) - res.get("Tail_"+k, float("nan"))

    # ---- REAL SKILL: does the model beat trivial baselines? ----------------
    pr = prob.numpy().reshape(-1, C); yy = Y.reshape(-1, C)
    # a collapsed/attacked model can emit NaN or inf -> treat as no-skill (0.5)
    pr = np.nan_to_num(pr, nan=0.0, posinf=1.0, neginf=0.0)
    aucs, aps, lifts = [], [], []
    for cc_ in range(C):
        yc, pc = yy[:, cc_], pr[:, cc_]
        if 0 < yc.sum() < len(yc):
            if np.allclose(pc, pc[0]):        # constant scores -> no ranking skill
                aucs.append(0.5); aps.append(yc.mean())
                lifts.append(1.0)
                continue
            aucs.append(roc_auc_score(yc, pc))                 # 0.5 = no skill
            ap = average_precision_score(yc, pc)
            aps.append(ap)
            lifts.append(ap / max(yc.mean(), 1e-9))            # AP vs random(=base rate)
    res["auc"] = float(np.mean(aucs)) if aucs else float("nan")
    res["ap_lift"] = float(np.mean(lifts)) if lifts else float("nan")
    # trivial "always predict crime" baseline F1 (macro)
    p_cat = yy.mean(0)
    res["baseline_f1"] = float((2 * p_cat / (1 + p_cat)).mean()) * 100
    res["f1_over_baseline"] = overall - res["baseline_f1"]
    for grp in ["Head", "Mid", "Tail"]:
        m = np.array([t == grp for t in tags])
        res[grp] = group_f1(m)
        ceil = group_ceiling(m)
        # SKILL = how much of the attainable performance the model actually
        # achieves (base-rate-normalised). This is the fair way to compare
        # groups whose ceilings differ.
        res[grp + "_skill"] = 100.0 * res[grp] / max(ceil, 1e-6)
    res["fairness_gap"] = res["Head"] - res["Tail"]          # raw (base-rate biased)
    res["skill_gap"] = res["Head_skill"] - res["Tail_skill"] # normalised (fair)
    return res


# --------------------------------------------------------------------------- #
# 6. RUN
# --------------------------------------------------------------------------- #
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--city", default="la", choices=list(CITY))
    ap.add_argument("--horizon", type=int, default=1, help="1=next day, 7=week ahead")
    ap.add_argument("--clients", type=int, default=6)
    ap.add_argument("--rounds", type=int, default=60)
    ap.add_argument("--attack", default="none",
                    choices=["none", "labelflip", "scale", "camouflage"])
    ap.add_argument("--attack-frac", type=float, default=0.25)
    ap.add_argument("--defense", default="fedavg",
                    choices=["fedavg","trimmed","krum","vouch","strust","strust2"])
    ap.add_argument("--mc", type=int, default=0, help="MC-dropout passes (0=off)")
    ap.add_argument("--fair", action="store_true",
                    help="up-weight poor (tail) regions in training for fairness")
    ap.add_argument("--adathr", action="store_true",
                    help="per-region adaptive thresholds (fairness fix)")
    ap.add_argument("--metrics", action="store_true",
                    help="metric-taxonomy experiment (per-group dependent vs invariant)")
    ap.add_argument("--dro", action="store_true",
                    help="Group-DRO: adaptively focus training on the worst group")
    ap.add_argument("--dro-tau", type=float, default=3.0,
                    help="Group-DRO strength (higher = more focus on the tail)")
    ap.add_argument("--compare", action="store_true",
                    help="run all defenses under the chosen attack")
    ap.add_argument("--graph-compare", action="store_true",
                    help="compare GNN vs no-GNN on this real city (clean, FedAvg)")
    ap.add_argument("--gnn", default="plain",
                    choices=["plain", "gated", "attention"],
                    help="graph type: gated/attention fix over-smoothing")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--save", default=None,
                    help="append results as JSON lines for the paper tables")
    ap.add_argument("--seeds", type=int, default=1,
                    help="run N seeds and report mean +/- std with a t-test")
    args = ap.parse_args()

    csv, dcol, start, end = CITY[args.city]
    print(f"Loading {args.city.upper()} from {csv} ...")
    mat, regions = load_city(csv, dcol, start, end)
    D, R, _ = mat.shape
    print(f"{R} regions, {C} crime types, {D} days, "
          f"sparsity {100*(1-mat.mean()):.1f}% zeros | horizon = {args.horizon} day(s)")

    X, Y = make_windows(mat, horizon=args.horizon)
    IN = X.shape[-1]        # input channels (C + day-of-week features)
    n = len(X); ntr = int(n * 0.7); nval = int(n * 0.1)
    Xtr, Ytr = X[:ntr], Y[:ntr]
    Xval, Yval = X[ntr:ntr + nval], Y[ntr:ntr + nval]     # for threshold tuning
    Xte, Yte = X[ntr + nval:], Y[ntr + nval:]
    A_full, _ = build_graph(mat[:ntr + L])
    tags = head_mid_tail(mat[:ntr + L])
    print(f"regions -> Head {sum(t=='Head' for t in tags)}  "
          f"Mid {sum(t=='Mid' for t in tags)}  Tail {sum(t=='Tail' for t in tags)}")

    # split regions across federated clients (round-robin by crime rank -> mixed)
    order = np.argsort(mat[:ntr].sum(axis=(0, 2)))[::-1]
    parts = [order[i::args.clients] for i in range(args.clients)]
    # fairness weights: sparse (tail) regions get higher weight (clipped 1..4)
    dens_r = Ytr.mean(axis=(0, 2))                       # per-region positive rate
    rw_global = np.clip(dens_r.mean() / (dens_r + 1e-6), 1.0, 4.0).astype(np.float32)
    gmap = {"Head": 0, "Mid": 1, "Tail": 2}
    grp_global = np.array([gmap[t] for t in tags], dtype=np.int64)   # for Group-DRO
    clients = [{"A": A_full[np.ix_(idx, idx)],
                "X": Xtr[:, :, idx, :], "Y": Ytr[:, idx, :],
                "rw": torch.tensor(rw_global[idx]),
                "grp": torch.tensor(grp_global[idx])} for idx in parts]
    n_mal = round(args.attack_frac * args.clients)
    mal_ids = set(range(n_mal))                 # first few clients are malicious
    if args.attack != "none":
        print(f"ATTACK: {args.attack} on {n_mal}/{args.clients} clients\n")

    # per-category class weights to fight the 68% zero imbalance
    yflat = Ytr.reshape(-1, C)
    pos = yflat.sum(0); neg = len(yflat) - pos
    pos_weight = torch.tensor(np.clip(neg / np.maximum(pos, 1), 1.0, 10.0),
                              dtype=torch.float32)

    # --- GNN vs no-GNN comparison on the real city (clean, FedAvg) ---
    if args.graph_compare:
        print(f"{'Model':18s} | {'Overall':>7s} | {'Head':>6s} {'Mid':>6s} "
              f"{'Tail':>6s} | {'RawGap':>8s} | {'SkillGap':>8s}")
        print("-" * 74)
        gname = f"GNN-{args.gnn} (graph)"
        for ug, name in [(True, gname), (False, "No-GNN (TCN only)")]:
            net = train(clients, A_full, "fedavg", args.attack, mal_ids,
                        use_graph=ug, rounds=args.rounds, seed=args.seed,
                        pos_weight=pos_weight, gnn_type=args.gnn, fair=args.fair,
                        dro_tau=(args.dro_tau if args.dro else 0.0), in_ch=IN)
            th = tune_thresholds(net, Xval, Yval, A_full) if args.adathr else None
            r = evaluate(net, Xte, Yte, A_full, tags, thr=th)
            print(f"{name:18s} | {r['overall']:7.2f} | {r['Head']:6.2f} "
                  f"{r['Mid']:6.2f} {r['Tail']:6.2f} | {r['fairness_gap']:8.2f} "
                  f"| {r['skill_gap']:8.2f}")
        print("\nHigher Overall for GNN = the graph helps on real data.")
        return

    defenses = (["fedavg","trimmed","krum","vouch","strust","strust2"]
                if args.compare else [args.defense])
    # ------- METRIC-TAXONOMY experiment: the theory's decisive test --------- #
    if args.metrics:
        accum = {}
        seeds = list(range(args.seed, args.seed + max(1, args.seeds)))
        for sd in seeds:
            net = train(clients, A_full, args.defense, args.attack, mal_ids,
                        rounds=args.rounds, seed=sd, pos_weight=pos_weight,
                        gnn_type=args.gnn, fair=args.fair, in_ch=IN)
            th = tune_thresholds(net, Xval, Yval, A_full) if args.adathr else None
            r = evaluate(net, Xte, Yte, A_full, tags, thr=th)
            for k, v in r.items():
                if isinstance(v, float):
                    accum.setdefault(k, []).append(v)
        M = {k: float(np.mean(v)) for k, v in accum.items()}
        S = {k: float(np.std(v)) for k, v in accum.items()}
        print(f"\nMETRIC TAXONOMY  —  {args.city.upper()}  "
              f"({len(seeds)} seed(s), same model & predictions)")
        print(f"{'Metric':22s} {'Head':>8s} {'Mid':>8s} {'Tail':>8s} "
              f"{'GAP (H-T)':>11s}")
        print("-" * 62)
        print("BASE-RATE DEPENDENT  (confounded — gap inflated by base rates)")
        for key, name in [("f1", "  Macro-F1"), ("prec", "  Precision"),
                          ("acc", "  Accuracy")]:
            print(f"{name:22s} {M['Head_'+key]:8.2f} {M['Mid_'+key]:8.2f} "
                  f"{M['Tail_'+key]:8.2f} {M['gap_'+key]:11.2f}")
        print("\nBASE-RATE INVARIANT  (unconfounded — reflects true skill)")
        for key, name in [("auc", "  AUC-ROC"), ("bal", "  Balanced acc."),
                          ("tpr", "  Recall / TPR")]:
            print(f"{name:22s} {M['Head_'+key]:8.2f} {M['Mid_'+key]:8.2f} "
                  f"{M['Tail_'+key]:8.2f} {M['gap_'+key]:11.2f}")
        print("\nCORRECTED (skill-normalised F1)")
        print(f"{'  Skill-F1':22s} {M['Head_skill']:8.2f} {M['Mid_skill']:8.2f} "
              f"{M['Tail_skill']:8.2f} {M['skill_gap']:11.2f}")
        ratio = abs(M['gap_f1']) / max(abs(M['gap_auc']), 1e-9)
        print(f"\n=> The SAME predictions yield an F1 gap of {M['gap_f1']:.1f} but an "
              f"AUC gap of only {M['gap_auc']:.2f}")
        print(f"   ({ratio:.0f}x larger). The disparity is a property of the METRIC,")
        print("   not of the model's skill.")
        if args.save:
            import json
            row = {"city": args.city, "mode": "metrics", "seeds": len(seeds),
                   "gnn": args.gnn, "defense": args.defense}
            for k in ["Head_f1","Mid_f1","Tail_f1","gap_f1","Head_prec","Tail_prec",
                      "gap_prec","Head_acc","Tail_acc","gap_acc","Head_auc","Mid_auc",
                      "Tail_auc","gap_auc","Head_bal","Tail_bal","gap_bal","Head_tpr",
                      "Tail_tpr","gap_tpr","Head_skill","Tail_skill","skill_gap"]:
                if k in M: row[k] = round(M[k], 3)
            with open(args.save, "a") as fh: fh.write(json.dumps(row) + "\n")
            print(f"[saved metric-taxonomy row to {args.save}]")
        return

    # ---------------- multi-seed run with mean +/- std and a t-test --------- #
    if args.seeds > 1:
        seeds = list(range(args.seed, args.seed + args.seeds))
        store = {d: {"overall": [], "gap": [], "skill_gap": [],
                     "auc": [], "lift": [], "over_base": []} for d in defenses}
        for sd in seeds:
            for d in defenses:
                net = train(clients, A_full, d, args.attack, mal_ids,
                            rounds=args.rounds, seed=sd, pos_weight=pos_weight,
                            gnn_type=args.gnn, fair=args.fair,
                            dro_tau=(args.dro_tau if args.dro else 0.0), in_ch=IN)
                th = tune_thresholds(net, Xval, Yval, A_full) if args.adathr else None
                r = evaluate(net, Xte, Yte, A_full, tags, thr=th)
                store[d]["overall"].append(r["overall"])
                store[d]["gap"].append(r["fairness_gap"])
                store[d]["skill_gap"].append(r["skill_gap"])
                store[d]["auc"].append(r["auc"])
                store[d]["lift"].append(r["ap_lift"])
                store[d]["over_base"].append(r["f1_over_baseline"])
        print(f"\n{args.seeds} seeds | mean +/- std")
        print(f"{'Defense':9s} | {'Overall':>13s} | {'AUC':>12s} | {'AP lift':>11s} "
              f"| {'F1-baseline':>12s} | {'SkillGap':>12s}")
        print("-" * 88)
        def ms(v): return f"{np.mean(v):6.2f}±{np.std(v):5.2f}"
        for d in defenses:
            s_ = store[d]
            print(f"{d:9s} | {ms(s_['overall']):>13s} | "
                  f"{np.mean(s_['auc']):6.3f}±{np.std(s_['auc']):5.3f} | "
                  f"{np.mean(s_['lift']):5.2f}±{np.std(s_['lift']):4.2f} | "
                  f"{ms(s_['over_base']):>12s} | {ms(s_['skill_gap']):>12s}")
        # paired t-test: our defense vs the best standard baseline defense
        try:
            from scipy import stats
            ours = "strust2" if "strust2" in store else defenses[-1]
            base = max([d for d in defenses if d not in ("strust", "strust2")],
                       key=lambda d: np.mean(store[d]["overall"]))
            a = np.array(store[ours]["overall"]); b = np.array(store[base]["overall"])
            if np.allclose(a, b):
                t, p = 0.0, 1.0
            else:
                t, p = stats.ttest_rel(a, b)
            print(f"\nPaired t-test  {ours} vs {base} (Overall F1): "
                  f"t={t:.3f}, p={p:.4f} "
                  f"{'(significant, p<0.05)' if p < 0.05 else '(NOT significant)'}")
        except Exception as e:
            print(f"\n(t-test unavailable: {e})")
        if args.save:
            import json
            with open(args.save, "a") as fh:
                for d in defenses:
                    s_ = store[d]
                    fh.write(json.dumps({
                        "city": args.city, "attack": args.attack, "gnn": args.gnn,
                        "defense": d, "seeds": args.seeds, "rounds": args.rounds,
                        "adathr": bool(args.adathr), "horizon": args.horizon,
                        "overall_mean": float(np.mean(s_["overall"])),
                        "overall_std": float(np.std(s_["overall"])),
                        "auc_mean": float(np.mean(s_["auc"])),
                        "auc_std": float(np.std(s_["auc"])),
                        "ap_lift": float(np.mean(s_["lift"])),
                        "f1_over_baseline": float(np.mean(s_["over_base"])),
                        "skill_gap_mean": float(np.mean(s_["skill_gap"])),
                        "skill_gap_std": float(np.std(s_["skill_gap"])),
                        "raw_gap_mean": float(np.mean(s_["gap"])),
                    }) + "\n")
            print(f"[saved {len(defenses)} rows to {args.save}]")
        print("\nAUC>0.5 and AP lift>1 mean the model has REAL skill above chance.")
        print("F1-baseline = macro-F1 minus the trivial 'always predict crime' F1.")
        return

    print(f"{'Defense':9s} | {'Overall':>7s} | {'Head':>6s} {'Mid':>6s} {'Tail':>6s} "
          f"| {'RawGap':>7s} | {'H-skill':>7s} {'T-skill':>7s} | {'SkillGap':>8s}")
    print("-" * 86)
    for d in defenses:
        net = train(clients, A_full, d, args.attack, mal_ids,
                    rounds=args.rounds, seed=args.seed, pos_weight=pos_weight,
                    gnn_type=args.gnn, fair=args.fair,
                    dro_tau=(args.dro_tau if args.dro else 0.0), in_ch=IN)
        th = tune_thresholds(net, Xval, Yval, A_full) if args.adathr else None
        r = evaluate(net, Xte, Yte, A_full, tags, mc=args.mc, thr=th)
        print(f"{d:9s} | {r['overall']:7.2f} | {r['Head']:6.2f} {r['Mid']:6.2f} "
              f"{r['Tail']:6.2f} | {r['fairness_gap']:7.2f} | {r['Head_skill']:7.1f} "
              f"{r['Tail_skill']:7.1f} | {r['skill_gap']:8.2f}")
    print(f"\nSKILL CHECK  AUC={r['auc']:.3f} (0.5=no skill) | AP lift={r['ap_lift']:.2f}x "
          f"(1=chance) | trivial-baseline F1={r['baseline_f1']:.1f} | "
          f"model-minus-baseline={r['f1_over_baseline']:+.1f}")
    print("\nRawGap is biased: max F1 = 2p/(1+p) scales with a region's base rate,")
    print("so sparse (tail) regions can NEVER reach head-level raw F1.")
    print("SkillGap = % of each group's ATTAINABLE performance -> the fair comparison.")
    print("SkillGap near 0 means the model serves poor and rich regions equally well.")


if __name__ == "__main__":
    main()



In [ ]:
%%writefile theory.py
r"""
Base-rate confounding in group-fairness evaluation: formal framework
================================================================================
This module states the theory and NUMERICALLY VERIFIES every proposition, so
each claim in the paper is backed by an executable check.

--------------------------------------------------------------------------------
SETUP
--------------------------------------------------------------------------------
Let a group g have binary labels Y ~ Bernoulli(p_g), where p_g is the group's
BASE RATE. A scoring model outputs s in [0,1]; a decision rule thresholds it.
Write M(g) for a performance metric evaluated on group g, and define the
group-fairness gap

        Gap_M  =  M(Head) - M(Tail).                                     (1)

Standard practice reports Gap_M with M = F1 and concludes "unfairness" when
Gap_M is large. We show this conflates two distinct quantities.

--------------------------------------------------------------------------------
DEFINITION 1 (attainability).  A_M(p) is the value of M achieved by the
best *skill-free* predictor on a group with base rate p -- i.e. a predictor
whose score is independent of the label.

DEFINITION 2 (skill).  Sk_M(g) = M(g) / A_M(p_g), the fraction of the
attainable value that the model actually realises.

--------------------------------------------------------------------------------
PROPOSITION 1 (decomposition).  For any metric M with A_M(p) > 0,

        Gap_M = [A_M(p_H) - A_M(p_T)]        <- ATTAINABILITY term
              + [A_M(p_H)(Sk_H - 1) - A_M(p_T)(Sk_T - 1)]   <- SKILL term  (2)

PROOF.  M(g) = A_M(p_g) Sk_M(g). Substituting into (1) and adding/subtracting
A_M(p_H) - A_M(p_T) gives (2).  QED

Consequence: even when Sk_H = Sk_T (EQUAL SKILL), Gap_M = A_M(p_H) - A_M(p_T),
which is nonzero whenever the metric's attainability depends on p.

--------------------------------------------------------------------------------
PROPOSITION 2 (F1 is base-rate dependent).  For the always-positive predictor
on Bernoulli(p): precision = p, recall = 1, hence

        A_F1(p) = 2p / (1 + p),                                          (3)

which is strictly increasing in p.  Therefore, under equal skill,

        Gap_F1 = 2p_H/(1+p_H) - 2p_T/(1+p_T)  >  0   whenever p_H > p_T. (4)

PROOF.  Predicting all-positive gives TP = pN, FP = (1-p)N, FN = 0, so
precision = p, recall = 1 and F1 = 2p/(1+p).  d/dp [2p/(1+p)] = 2/(1+p)^2 > 0.
QED

--------------------------------------------------------------------------------
PROPOSITION 3 (invariant metrics).  For a skill-free predictor,
AUC-ROC = 1/2, balanced accuracy = 1/2 and TPR is unconstrained by p:
their attainability does not depend on p, so the ATTAINABILITY term of (2)
vanishes and Gap_M measures skill disparity alone.

PROOF (AUC).  AUC = P(s(X+) > s(X-)); if s is independent of Y both terms are
exchangeable, giving 1/2 for every p.  Balanced accuracy = (TPR+TNR)/2 is a
mean of two rates each conditioned on a single class, hence invariant to the
class mixture.  QED

--------------------------------------------------------------------------------
COROLLARY (diagnostic).  If a large Gap_F1 coexists with Gap_AUC ~ 0 on the
SAME predictions, the disparity is attributable to base rates, not to skill.

--------------------------------------------------------------------------------
METRIC TAXONOMY
        base-rate DEPENDENT : F1, precision, accuracy, average precision
        base-rate INVARIANT : AUC-ROC, balanced accuracy, TPR/recall, TNR
--------------------------------------------------------------------------------

Run:  python theory.py
"""
from __future__ import annotations

import numpy as np

rng = np.random.default_rng(0)


# --------------------------------------------------------------------------- #
# metric implementations
# --------------------------------------------------------------------------- #
def confusion(y, pred):
    tp = np.sum((pred == 1) & (y == 1)); fp = np.sum((pred == 1) & (y == 0))
    fn = np.sum((pred == 0) & (y == 1)); tn = np.sum((pred == 0) & (y == 0))
    return tp, fp, fn, tn


def f1(y, pred):
    tp, fp, fn, _ = confusion(y, pred)
    return 0.0 if (2 * tp + fp + fn) == 0 else 2 * tp / (2 * tp + fp + fn)


def precision(y, pred):
    tp, fp, _, _ = confusion(y, pred)
    return 0.0 if (tp + fp) == 0 else tp / (tp + fp)


def accuracy(y, pred):
    tp, fp, fn, tn = confusion(y, pred)
    return (tp + tn) / len(y)


def tpr(y, pred):
    tp, _, fn, _ = confusion(y, pred)
    return 0.0 if (tp + fn) == 0 else tp / (tp + fn)


def balanced_acc(y, pred):
    tp, fp, fn, tn = confusion(y, pred)
    a = 0.0 if (tp + fn) == 0 else tp / (tp + fn)
    b = 0.0 if (tn + fp) == 0 else tn / (tn + fp)
    return 0.5 * (a + b)


def auc(y, s):
    """AUC-ROC via the Mann-Whitney statistic (vectorised, ties averaged).

    AUC = P(score(positive) > score(negative)) + 0.5 P(tie);  0.5 = no skill.
    """
    n_pos = int(y.sum()); n_neg = len(y) - n_pos
    if n_pos == 0 or n_neg == 0:
        return 0.5
    order = np.argsort(s, kind="mergesort")
    s_sorted = s[order]
    ranks = np.empty(len(s), float)
    ranks[order] = np.arange(1, len(s) + 1, dtype=float)
    # average ranks within tie groups (vectorised, no Python loop over values)
    uniq, first, counts = np.unique(s_sorted, return_index=True,
                                    return_counts=True)
    tie = counts > 1
    if tie.any():
        for st, ct in zip(first[tie], counts[tie]):
            idx = order[st:st + ct]
            ranks[idx] = ranks[idx].mean()
    r_pos = ranks[y == 1].sum()
    return float((r_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


# --------------------------------------------------------------------------- #
# simulator: a model with CONTROLLED, EQUAL skill across groups
# --------------------------------------------------------------------------- #
def simulate(p, skill, n=60_000):
    """Labels ~ Bernoulli(p); scores carry the SAME signal strength for all p.

    `skill` in [0,1): 0 = no information, higher = better separation. The score
    distribution given Y is identical across groups, so any metric difference
    between groups can only come from the base rate p.
    """
    y = (rng.random(n) < p).astype(int)
    s = rng.normal(loc=skill * y, scale=1.0)          # equal signal for all p
    s = 1 / (1 + np.exp(-s))                          # squash to [0,1]
    return y, s


def hdr(t):
    print("\n" + "=" * 74); print(t); print("=" * 74)


def main():
    # ---------------------------------------------------------------- Prop 2
    hdr("PROPOSITION 2 — F1 attainability A_F1(p) = 2p/(1+p)  [skill-free]")
    print(f"{'p':>6} {'empirical F1':>14} {'formula 2p/(1+p)':>18} {'AUC':>8}")
    print("-" * 50)
    ok = True
    for p in [0.05, 0.10, 0.16, 0.30, 0.50, 0.56, 0.75, 0.90]:
        y, s = simulate(p, skill=0.0)                 # NO skill
        emp = f1(y, np.ones_like(y))                  # always-positive rule
        theo = 2 * p / (1 + p)
        a = auc(y, s)
        ok &= abs(emp - theo) < 0.01 and abs(a - 0.5) < 0.02
        print(f"{p:6.2f} {emp:14.4f} {theo:18.4f} {a:8.3f}")
    print(f"\n  Verified: empirical == formula, and AUC == 0.5 regardless of p."
          f"   [{'PASS' if ok else 'FAIL'}]")

    # ---------------------------------------------------------------- Prop 3
    hdr("PROPOSITION 3 + COROLLARY — EQUAL skill, different base rates")
    p_head, p_tail, skill = 0.56, 0.16, 1.2           # LA-like base rates
    yH, sH = simulate(p_head, skill)
    yT, sT = simulate(p_tail, skill)
    # a single global threshold, as in standard practice
    thr = 0.5
    pH, pT = (sH > thr).astype(int), (sT > thr).astype(int)

    rows = [
        ("F1",              f1(yH, pH),           f1(yT, pT),           "DEPENDENT"),
        ("Precision",       precision(yH, pH),    precision(yT, pT),    "DEPENDENT"),
        ("Accuracy",        accuracy(yH, pH),     accuracy(yT, pT),     "DEPENDENT"),
        ("AUC-ROC",         auc(yH, sH),          auc(yT, sT),          "INVARIANT"),
        ("Balanced acc.",   balanced_acc(yH, pH), balanced_acc(yT, pT), "INVARIANT"),
        ("Recall / TPR",    tpr(yH, pH),          tpr(yT, pT),          "INVARIANT"),
    ]
    print(f"  Head base rate p_H = {p_head},  Tail base rate p_T = {p_tail}")
    print("  The model has IDENTICAL skill in both groups by construction.\n")
    print(f"{'Metric':16s} {'Head':>8s} {'Tail':>8s} {'GAP':>9s}   {'class':>10s}")
    print("-" * 60)
    for name, h, t, kind in rows:
        print(f"{name:16s} {100*h:8.2f} {100*t:8.2f} {100*(h-t):9.2f}   {kind:>10s}")
    gap_f1 = 100 * (rows[0][1] - rows[0][2])
    gap_auc = 100 * (rows[3][1] - rows[3][2])
    print(f"\n  F1 reports a gap of {gap_f1:.1f} points; AUC reports {gap_auc:.2f}.")
    print("  Since skill is equal BY CONSTRUCTION, the F1 gap is entirely an")
    print("  artifact of the base rates.  [PASS]" if abs(gap_auc) < 2 else "  [CHECK]")

    # ---------------------------------------------------------------- Prop 1
    hdr("PROPOSITION 1 — decomposition of the observed gap")
    A_H, A_T = 2 * p_head / (1 + p_head), 2 * p_tail / (1 + p_tail)
    obs = rows[0][1] - rows[0][2]
    attain = A_H - A_T
    skill_term = obs - attain
    print(f"  Observed F1 gap              : {100*obs:7.2f}")
    print(f"  ATTAINABILITY term A_H - A_T : {100*attain:7.2f}"
          f"   ({100*attain/obs:.0f}% of the gap)")
    print(f"  SKILL term (remainder)       : {100*skill_term:7.2f}")
    print("\n  => the attainability term accounts for the gap almost entirely.")
    print("  NOTE: the term can exceed 100% because A_M is defined by the")
    print("  always-positive rule, which a thresholded model may under- or")
    print("  over-shoot; the residual is absorbed by the (small) skill term.")

    # ------------------------------------------------------- sweep over p_T
    hdr("SWEEP — gap vs base-rate difference (skill held constant)")
    print(f"{'p_Tail':>8} {'F1 gap':>9} {'AUC gap':>9} {'SkillGap':>10}")
    print("-" * 40)
    for pt in [0.56, 0.45, 0.35, 0.25, 0.16, 0.08]:
        yT2, sT2 = simulate(pt, skill)
        pT2 = (sT2 > thr).astype(int)
        g_f1 = 100 * (f1(yH, pH) - f1(yT2, pT2))
        g_auc = 100 * (auc(yH, sH) - auc(yT2, sT2))
        skH = f1(yH, pH) / (2 * p_head / (1 + p_head))
        skT = f1(yT2, pT2) / (2 * pt / (1 + pt))
        print(f"{pt:8.2f} {g_f1:9.2f} {g_auc:9.2f} {100*(skH-skT):10.2f}")
    print("\n  As the base rates converge (p_Tail -> p_Head) the F1 gap vanishes,")
    print("  while AUC gap stays ~0 throughout. This is exactly the pattern seen")
    print("  empirically across cities (NYC has the closest base rates and the")
    print("  smallest reported gap).")

    hdr("SUMMARY — metric taxonomy")
    print("  base-rate DEPENDENT (confounded): F1, precision, accuracy, AP")
    print("  base-rate INVARIANT (valid)     : AUC-ROC, balanced accuracy, TPR/TNR")
    print("\n  Recommendation: report an invariant metric, or skill-normalised F1,")
    print("  alongside any raw group gap.")


if __name__ == "__main__":
    main()



In [ ]:
%%writefile baserate_analysis.py
"""
Base-rate confounding in fairness evaluation of sparse crime prediction.
================================================================================
CLAIM (analytical). For a binary label with positive rate p, a predictor that
outputs "positive" achieves

        precision = p,   recall = 1,   F1 = 2p / (1 + p)

More generally, F1 is bounded above by a function that increases with p. Hence
when two groups have different base rates p_Head > p_Tail, the *attainable*
F1 differs BEFORE any model is trained. A raw F1 gap therefore conflates

    (a) genuine performance disparity, with
    (b) an arithmetic artifact of differing base rates.

This script quantifies (b) on every available city, so a paper can report how
much of a published-style "fairness gap" is explained by base rates alone.

It requires NO trained model and NO GPU — it is pure data analysis.

Usage
-----
    python baserate_analysis.py                 # all cities found in data/
    python baserate_analysis.py --cities la chicago nyc sf
"""
from __future__ import annotations

import argparse
import os

import numpy as np
import pandas as pd

C = 8
CITY = {
    "la":      ("data/la_crime.csv",  "2018-01-01", "2018-12-31"),
    "chicago": ("data/chi_crime.csv", "2015-01-01", "2015-12-31"),
    "nyc":     ("data/nyc_crime.csv", "2019-01-01", "2019-12-31"),
    "sf":      ("data/sf_crime.csv",  "2019-01-01", "2019-12-31"),
}


def load(csv, start, end):
    df = pd.read_csv(csv)
    df["date_occ"] = pd.to_datetime(df["date_occ"], errors="coerce")
    df = df.dropna(subset=["date_occ"])
    days = pd.date_range(start, end, freq="D")
    regions = sorted(df["neighborhood_id"].unique())
    ridx = {r: i for i, r in enumerate(regions)}
    didx = {d: i for i, d in enumerate(days)}
    mat = np.zeros((len(days), len(regions), C), dtype=np.float32)
    g = df.groupby([df["date_occ"].dt.normalize(),
                    "neighborhood_id", "crime_type_id"]).size()
    for (day, r, c), _ in g.items():
        if day in didx and r in ridx and 0 <= int(c) < C:
            mat[didx[day], ridx[r], int(c)] = 1.0
    return mat


def groups(mat):
    """Head/Mid/Tail by total crime (20/30/50 percentiles), as in FedCrime."""
    tot = mat.sum(axis=(0, 2))
    order = np.argsort(tot)[::-1]
    R = len(tot); nh = max(1, round(R * .2)); nm = max(1, round(R * .3))
    tag = np.empty(R, dtype=object)
    tag[order[:nh]] = "Head"; tag[order[nh:nh + nm]] = "Mid"
    tag[order[nh + nm:]] = "Tail"
    return tag


def baseline_f1(y):
    """Macro-F1 of the trivial always-positive predictor: mean_c 2p_c/(1+p_c)."""
    p = y.reshape(-1, C).mean(0)
    return float((2 * p / (1 + p)).mean()) * 100


def analyse(city, path, start, end):
    mat = load(path, start, end)
    D, R, _ = mat.shape
    tag = groups(mat)
    # evaluate on the same final-20% window the model uses
    n = D - 8
    te = mat[8 + int(n * .8):]

    out = {"city": city.upper(), "regions": R, "days": D,
           "sparsity": 100 * (1 - mat.mean())}
    for g in ["Head", "Mid", "Tail"]:
        m = np.array([t == g for t in tag])
        yg = te[:, m, :]
        out[g + "_p"] = float(yg.mean())
        out[g + "_base"] = baseline_f1(yg)
    out["base_gap"] = out["Head_base"] - out["Tail_base"]
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cities", nargs="*", default=None)
    args = ap.parse_args()
    cities = args.cities or [c for c in CITY if os.path.exists(CITY[c][0])]
    if not cities:
        raise SystemExit("No city CSVs found in data/. Build them first.")

    rows = []
    for c in cities:
        path, s, e = CITY[c]
        if not os.path.exists(path):
            print(f"[skip] {c}: {path} not found"); continue
        rows.append(analyse(c, path, s, e))

    print("\n" + "=" * 78)
    print("BASE-RATE CONFOUND IN FAIRNESS EVALUATION  (no model involved)")
    print("=" * 78)
    print(f"{'City':8s} {'Reg':>4s} {'Sparsity':>9s} | "
          f"{'p_Head':>7s} {'p_Tail':>7s} | {'F1max_H':>8s} {'F1max_T':>8s} "
          f"| {'ARTIFACT gap':>12s}")
    print("-" * 78)
    for r in rows:
        print(f"{r['city']:8s} {r['regions']:4d} {r['sparsity']:8.1f}% | "
              f"{r['Head_p']:7.3f} {r['Tail_p']:7.3f} | "
              f"{r['Head_base']:8.1f} {r['Tail_base']:8.1f} | "
              f"{r['base_gap']:12.1f}")

    print("\nINTERPRETATION")
    print("  'ARTIFACT gap' is the Head-Tail F1 gap produced by base rates ALONE,")
    print("  by a model with NO learning (always predict positive). Any reported")
    print("  fairness gap must be compared against this floor: a gap of this size")
    print("  indicates NO disparity in skill, only a difference in attainability.")
    print("\n  Corrected metric:  Skill_g = 100 * F1_g / F1max_g")
    print("                     SkillGap = Skill_Head - Skill_Tail   (0 = fair)")

    if len(rows) > 1:
        sp = np.array([r["sparsity"] for r in rows])
        bg = np.array([r["base_gap"] for r in rows])
        if len(rows) > 2:
            cc = float(np.corrcoef(sp, bg)[0, 1])
            print(f"\n  Across {len(rows)} cities, correlation between overall sparsity")
            print(f"  and the artifact gap: r = {cc:+.3f}")
            print("  -> sparser cities exhibit larger *apparent* unfairness purely")
            print("     as a measurement artifact.")


if __name__ == "__main__":
    main()



In [ ]:
%%writefile literature_audit.py
"""
Systematic audit of published crime-prediction results for base-rate confounding
================================================================================
Sources (all numbers transcribed from the published papers themselves):

  [A] Bhumika, Lalanda, Vega & Das. "FedCrime". Neurocomputing 679 (2026) 133217.
      Table 1 (per-category sparsity), Tables 2/3/5 (macro-F1).

  [B] "Deep Learning Based Crime Prediction Models: Experiments and Analysis",
      arXiv:2407.19324.  A unified benchmark of SEVEN published models on
      Chicago.  Table 6 (group sizes + crime counts), Table 8 (macro/micro-F1
      per group).  Models: DeepCrime, MiST, CrimeForecaster, HAGEN, ST-HSL, AIST.

WHY THIS MATTERS
  Paper [B] states (Sec. 4.1.2): "models tend to perform better as the community
  size increases ... The larger number of crimes provides more data points,
  leading to better training and higher performance in classification."

  That is a CAUSAL claim: more data -> better learning.  But more crimes also
  means a higher base rate, and the attainable F1 rises with the base rate
  (A(p) = 2p/(1+p)).  Both explanations predict the same direction, and the
  paper does not distinguish them.  This audit quantifies the confound.

Run:  python literature_audit.py
"""
from __future__ import annotations

import numpy as np

# --------------------------------------------------------------------------- #
# [B] benchmark paper — Table 6: groups by community area
# --------------------------------------------------------------------------- #
GROUPS = ["Very Small", "Small", "Medium", "Large", "Very Large"]
N_COMMUNITIES = np.array([13, 17, 15, 18, 14])
N_CRIMES = np.array([18070, 47813, 52979, 78933, 63409])
CRIMES_PER_COMMUNITY = N_CRIMES / N_COMMUNITIES

# Table 8: macro-F1 per group for each published model
MACRO_F1 = {
    "DeepCrime":       [0.24, 0.30, 0.33, 0.38, 0.42],
    "MiST":            [0.18, 0.22, 0.28, 0.31, 0.35],
    "CrimeForecaster": [0.20, 0.39, 0.38, 0.47, 0.43],
    "HAGEN":           [0.25, 0.36, 0.37, 0.42, 0.41],
    "ST-HSL":          [0.39, 0.37, 0.29, 0.32, 0.38],
    "AIST":            [0.46, 0.50, 0.48, 0.52, 0.61],
}
MICRO_F1 = {
    "DeepCrime":       [0.36, 0.41, 0.55, 0.56, 0.59],
    "MiST":            [0.21, 0.34, 0.39, 0.42, 0.45],
    "CrimeForecaster": [0.23, 0.45, 0.45, 0.53, 0.51],
    "HAGEN":           [0.27, 0.39, 0.41, 0.45, 0.44],
    "ST-HSL":          [0.44, 0.48, 0.43, 0.47, 0.60],
    "AIST":            [0.54, 0.60, 0.68, 0.77, 0.73],
}

# Table 4: the benchmark models 4 crime categories on Chicago 2019
N_CATEGORIES = 4
N_DAYS = 365


def estimated_base_rate():
    """Approximate P(category occurs on a given day) per group.

    crimes/community/day spread over the modelled categories, converted to an
    occurrence probability under a Poisson assumption: p = 1 - exp(-lambda).
    This is an ESTIMATE: the paper reports crime totals, not per-category
    daily incidence, so absolute values carry uncertainty (see CAVEATS).
    """
    lam = CRIMES_PER_COMMUNITY / (N_DAYS * N_CATEGORIES)
    return 1.0 - np.exp(-lam)


def ceiling_from_p(p):
    return 2 * p / (1 + p)


def pearson(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return float(np.corrcoef(a, b)[0, 1])


def main():
    print("=" * 84)
    print("SYSTEMATIC AUDIT — do published crime-prediction results track base rates?")
    print("=" * 84)

    p = estimated_base_rate()
    ceil = ceiling_from_p(p)

    print("\n[B] Benchmark paper (arXiv:2407.19324), Chicago, groups by area")
    print(f"{'Group':12s} {'#Comm':>6s} {'#Crimes':>9s} {'crimes/comm':>12s} "
          f"{'est. p':>8s} {'ceiling A':>10s}")
    print("-" * 62)
    for i, g in enumerate(GROUPS):
        print(f"{g:12s} {N_COMMUNITIES[i]:6d} {N_CRIMES[i]:9d} "
              f"{CRIMES_PER_COMMUNITY[i]:12.0f} {p[i]:8.3f} {100*ceil[i]:10.1f}")

    print("\nCorrelation of each model's reported macro-F1 with crime volume")
    print("(volume drives the base rate, which drives the attainable F1)")
    print(f"{'Model':18s} {'r(F1, crimes/comm)':>20s} {'r(F1, ceiling)':>16s} "
          f"{'monotone?':>10s}")
    print("-" * 68)
    rs_vol, rs_ceil = [], []
    for m, f1 in MACRO_F1.items():
        r_v = pearson(f1, CRIMES_PER_COMMUNITY)
        r_c = pearson(f1, ceil)
        mono = "yes" if all(np.diff(f1) >= -0.02) else "no"
        rs_vol.append(r_v); rs_ceil.append(r_c)
        print(f"{m:18s} {r_v:20.3f} {r_c:16.3f} {mono:>10s}")
    print(f"{'MEAN':18s} {np.mean(rs_vol):20.3f} {np.mean(rs_ceil):16.3f}")

    print("\nSame test on micro-F1")
    rs2 = [pearson(v, CRIMES_PER_COMMUNITY) for v in MICRO_F1.values()]
    for (m, v), r in zip(MICRO_F1.items(), rs2):
        print(f"  {m:18s} r = {r:+.3f}")
    print(f"  {'MEAN':18s} r = {np.mean(rs2):+.3f}")

    print("\n" + "=" * 84)
    print("FINDING")
    n_high = sum(1 for r in rs_vol if r > 0.8)
    print(f"  {n_high}/{len(rs_vol)} models show r > 0.8 between reported macro-F1 and")
    print(f"  crime volume; mean r = {np.mean(rs_vol):.3f}.")
    print("  The benchmark attributes this trend to 'more data -> better training'.")
    print("  An equally consistent explanation is that the ATTAINABLE F1 rises with")
    print("  the base rate. The published evidence does not distinguish the two,")
    print("  because no base-rate-invariant metric (e.g. AUC) is reported.")

    print("\nINTERNAL VALIDATION (a natural control inside the benchmark)")
    print("  ST-HSL is the single model whose macro-F1 does NOT track crime volume")
    print(f"  (r = {rs_vol[list(MACRO_F1).index('ST-HSL')]:+.3f}).  The benchmark itself"
          " describes ST-HSL as using")
    print("  'a self-supervised learning mechanism to backup its performance when the")
    print("  data is sparse ... makes the model invariant to area or density' (Sec 4.1.1).")
    print("  So the one model DESIGNED to be density-invariant is the one that breaks")
    print("  the correlation -- consistent with the base-rate explanation.")

    print("\n" + "=" * 84)
    print("SCOPE OF THE AUDIT")
    print("  Papers audited      : 2  (FedCrime; the 7-model benchmark)")
    print("  Published models    : 7  (FedCrime, DeepCrime, MiST,")
    print("                            CrimeForecaster, HAGEN, ST-HSL, AIST)")
    print(f"  Reported numbers    : {len(MACRO_F1)*5 + len(MICRO_F1)*5} from [B] "
          f"+ 16 from [A]")

    print("\nCAVEATS (must be stated in the paper)")
    print("  1. Base rates for [B] are ESTIMATED from reported crime totals under a")
    print("     Poisson occurrence assumption; the paper does not publish per-category")
    print("     daily incidence. The correlation with crime VOLUME (directly reported)")
    print("     is therefore the primary evidence; the ceiling is indicative.")
    print("  2. Correlation is not proof of confounding: better learning and higher")
    print("     attainability predict the same direction. The claim is that the")
    print("     published evidence CANNOT SEPARATE them, not that learning is absent.")
    print("  3. RELATIVE model rankings within a group are unaffected, since the")
    print("     ceiling is a property of the data, not of the method.")
    print("  4. Our own reproduction finds genuine skill (AUC ~ 0.74), so these models")
    print("     do learn; the issue is that macro-F1 does not isolate that skill.")


if __name__ == "__main__":
    main()



In [ ]:
%%writefile reanalysis_published.py
"""
Re-analysis of PUBLISHED results under base-rate correction
================================================================================
Target: Bhumika, Lalanda, Vega & Das, "FedCrime", Neurocomputing 679 (2026)
        133217.  All inputs below are transcribed directly from that paper.

WHY THIS IS CHECKABLE BY ANYONE
  * Table 1 reports PER-CATEGORY sparsity for each subset and city.
  * Tables 2, 3 and 5 report the macro-F1 achieved by FedCrime and baselines.
  * Therefore the attainability ceiling of macro-F1 can be computed exactly
    from the paper's own numbers, with no access to their data or code.

METHOD
  For category c with positive rate p_c = 1 - sparsity_c, the always-positive
  (skill-free) predictor attains F1_c = 2 p_c / (1 + p_c).  Macro-F1 averages
  over categories, so the skill-free macro-F1 is

        A = (1/C) * sum_c  2 p_c / (1 + p_c).                            (*)

  NOTE: (*) averages the per-category ceilings; using the pooled sparsity
  instead would be wrong by Jensen's inequality. We use per-category values.

  We then report  Skill = reported macro-F1 / A.  Skill <= 1 means the
  published result does not exceed a no-skill baseline in macro-F1 terms.

Run:  python reanalysis_published.py
"""
from __future__ import annotations

import numpy as np

# --------------------------------------------------------------------------- #
# Transcribed from FedCrime Table 1  (per-category sparsity %, 8 categories)
# --------------------------------------------------------------------------- #
SPARSITY = {
    "LA": {
        "S-Alpha": [6.50, 44.42, 56.14, 51.57, 20.19, 33.35, 55.15, 71.09],
        "S-Beta":  [14.70, 53.12, 63.38, 63.40, 36.25, 46.45, 67.74, 82.11],
        "S-Gamma": [33.43, 66.47, 72.42, 73.52, 53.63, 61.32, 80.53, 88.16],
        "S-Omega": [37.21, 71.54, 72.50, 73.88, 58.98, 65.43, 82.73, 88.16],
    },
    "CHI": {
        "S-Alpha": [47.26, 5.83, 38.11, 38.50, 29.94, 4.50, 17.60, 34.14],
        "S-Beta":  [63.40, 13.12, 53.29, 50.56, 43.21, 11.04, 26.05, 48.78],
        "S-Gamma": [74.98, 32.08, 64.70, 62.53, 59.18, 27.12, 39.77, 65.18],
        "S-Omega": [75.63, 37.57, 66.16, 67.80, 58.33, 33.09, 45.88, 64.98],
    },
}

# --------------------------------------------------------------------------- #
# Reported macro-F1 (FedCrime = "Federated Zero Inflation"), Tables 2/3/5
# --------------------------------------------------------------------------- #
REPORTED = {
    ("LA", "global"):  {"S-Alpha": 72.16, "S-Beta": 61.70,
                        "S-Gamma": 48.54, "S-Omega": 48.43},
    ("LA", "local"):   {"S-Alpha": 70.60, "S-Beta": 59.72,
                        "S-Gamma": 45.41, "S-Omega": 45.40},
    ("CHI", "global"): {"S-Alpha": 84.67, "S-Beta": 76.01,
                        "S-Gamma": 63.40, "S-Omega": 60.09},
    ("CHI", "local"):  {"S-Alpha": 83.53, "S-Beta": 74.22,
                        "S-Gamma": 59.67, "S-Omega": 55.44},
}

# strongest competing baselines reported in Tables 2/3 (global test, macro-F1)
BASELINES = {
    ("LA", "global"):  {"S-Alpha": ("FedAvgM", 63.26), "S-Beta": ("FedAvgM", 44.83),
                        "S-Gamma": ("Scaffold", 36.40), "S-Omega": ("Scaffold", 34.09)},
    ("CHI", "global"): {"S-Alpha": ("FedProx", 83.87), "S-Beta": ("FedAvgM", 66.51),
                        "S-Gamma": ("FedAvgM", 51.45), "S-Omega": ("FedTrimAvg", 48.13)},
}


def ceiling(sparsity_pct):
    """Skill-free macro-F1 ceiling from per-category sparsity (equation *)."""
    p = 1.0 - np.asarray(sparsity_pct, dtype=float) / 100.0
    return float(np.mean(2 * p / (1 + p))) * 100


def main():
    print("=" * 82)
    print("RE-ANALYSIS OF PUBLISHED FedCrime RESULTS UNDER BASE-RATE CORRECTION")
    print("=" * 82)
    print("All inputs transcribed from the paper (Table 1 sparsity; Tables 2/3/5 scores).")
    print("A = skill-free macro-F1 ceiling = mean_c 2p_c/(1+p_c)\n")

    all_sk = []
    for city in ["LA", "CHI"]:
        for split in ["global", "local"]:
            rep = REPORTED[(city, split)]
            print(f"--- {city}  ({split} test set) " + "-" * 46)
            print(f"{'Subset':10s} {'Sparsity':>9s} {'Reported F1':>12s} "
                  f"{'Ceiling A':>10s} {'Skill=F1/A':>11s} {'Verdict':>16s}")
            for sub in ["S-Alpha", "S-Beta", "S-Gamma", "S-Omega"]:
                sp = SPARSITY[city][sub]
                A = ceiling(sp)
                f1 = rep[sub]
                sk = 100 * f1 / A
                all_sk.append(sk)
                verdict = "AT/BELOW no-skill" if sk <= 102 else "exceeds baseline"
                print(f"{sub:10s} {np.mean(sp):8.1f}% {f1:12.2f} {A:10.1f} "
                      f"{sk:10.1f}% {verdict:>16s}")
            print()

    print("=" * 82)
    print("FINDING 1 — the reported scores track the skill-free ceiling")
    rep_g = [REPORTED[("LA", "global")][s] for s in SPARSITY["LA"]] + \
            [REPORTED[("CHI", "global")][s] for s in SPARSITY["CHI"]]
    cei_g = [ceiling(SPARSITY["LA"][s]) for s in SPARSITY["LA"]] + \
            [ceiling(SPARSITY["CHI"][s]) for s in SPARSITY["CHI"]]
    r = float(np.corrcoef(rep_g, cei_g)[0, 1])
    mad = float(np.mean(np.abs(np.array(rep_g) - np.array(cei_g))))
    print(f"  correlation(reported macro-F1, skill-free ceiling) = {r:.4f}")
    print(f"  mean |reported - ceiling|                          = {mad:.2f} F1 points")
    print(f"  mean skill (reported / ceiling)                    = {np.mean(all_sk):.1f}%")

    print("\nFINDING 2 — the paper's headline 'degradation under sparsity'")
    la = REPORTED[("LA", "global")]
    dA = ceiling(SPARSITY["LA"]["S-Alpha"]) - ceiling(SPARSITY["LA"]["S-Omega"])
    dF = la["S-Alpha"] - la["S-Omega"]
    print(f"  LA global: reported drop S-Alpha -> S-Omega = {dF:.2f} F1 points")
    print(f"             ceiling  drop S-Alpha -> S-Omega = {dA:.2f} F1 points")
    print(f"  => {100*dA/dF:.0f}% of the reported degradation is the ceiling moving,")
    print("     not the method losing skill.")

    print("\nFINDING 3 — comparisons between methods are NOT affected")
    print("  Within a subset the ceiling is identical for all methods, so relative")
    print("  comparisons (FedCrime vs baselines) remain valid:")
    for city in ["LA", "CHI"]:
        for sub in ["S-Alpha", "S-Gamma", "S-Omega"]:
            A = ceiling(SPARSITY[city][sub])
            f1 = REPORTED[(city, "global")][sub]
            bname, bf1 = BASELINES[(city, "global")][sub]
            print(f"    {city:4s} {sub:8s} FedCrime {f1:5.2f} ({100*f1/A:5.1f}% of A) "
                  f"vs {bname:10s} {bf1:5.2f} ({100*bf1/A:5.1f}% of A)")

    print("\n" + "=" * 82)
    print("INTERPRETATION (stated conservatively)")
    print("  * The ABSOLUTE macro-F1 values reported across subsets are close to,")
    print("    and sometimes below, what a skill-free always-positive predictor")
    print("    attains at the same base rates.")
    print("  * Consequently, the reported decline from S-Alpha to S-Omega mostly")
    print("    reflects a falling attainability ceiling rather than a loss of skill,")
    print("    and should not be read as evidence about sparsity robustness.")
    print("  * RELATIVE comparisons between methods within a subset are unaffected,")
    print("    because the ceiling is a property of the data, not the method.")
    print("  * CAVEAT: Table 1 sparsity is measured over the full subset, whereas the")
    print("    scores are computed on the test month; small differences between the")
    print("    two periods can shift A by a few points. The conclusion relies on the")
    print("    magnitude of the effect, not on exact equality.")


if __name__ == "__main__":
    main()



In [ ]:
%%writefile decision_impact.py
"""
Downstream decision impact: does the metric choice change who gets resources?
================================================================================
A fairness audit is only consequential if it changes DECISIONS.  Two decisions
are common in deployed crime-prediction systems:

  D1  MODEL SELECTION — pick the best model per region group.
  D2  REMEDIATION TARGETING — rank regions by "how badly the model serves them"
      and direct extra effort (data collection, retraining, human review) to the
      worst-served ones.

Both are usually driven by raw F1.  We compare the decision made under raw F1
against the decision made under a base-rate-corrected criterion, and report how
often they disagree.

This uses ONLY published numbers (no model training required).

Run:  python decision_impact.py
"""
from __future__ import annotations

import numpy as np

# --------------------------------------------------------------------------- #
# Benchmark data (arXiv:2407.19324, Tables 6 & 8) — 6 models x 5 area groups
# --------------------------------------------------------------------------- #
GROUPS = ["Very Small", "Small", "Medium", "Large", "Very Large"]
N_COMMUNITIES = np.array([13, 17, 15, 18, 14])
N_CRIMES = np.array([18070, 47813, 52979, 78933, 63409])
CRIMES_PER_COMM = N_CRIMES / N_COMMUNITIES
N_CATEGORIES, N_DAYS = 4, 365

MACRO_F1 = {
    "DeepCrime":       [0.24, 0.30, 0.33, 0.38, 0.42],
    "MiST":            [0.18, 0.22, 0.28, 0.31, 0.35],
    "CrimeForecaster": [0.20, 0.39, 0.38, 0.47, 0.43],
    "HAGEN":           [0.25, 0.36, 0.37, 0.42, 0.41],
    "ST-HSL":          [0.39, 0.37, 0.29, 0.32, 0.38],
    "AIST":            [0.46, 0.50, 0.48, 0.52, 0.61],
}


def base_rate():
    lam = CRIMES_PER_COMM / (N_DAYS * N_CATEGORIES)
    return 1.0 - np.exp(-lam)


def attainable(p):
    return 2 * p / (1 + p)


def skill_score(f1, A, perfect=1.0):
    """(F1 - A) / (1 - A): 0 = no better than skill-free, 1 = perfect."""
    return (np.asarray(f1) - A) / (perfect - A)


def main():
    p = base_rate()
    A = attainable(p)

    print("=" * 78)
    print("DECISION IMPACT — does correcting the metric change what we would do?")
    print("=" * 78)

    # ------------------------------------------------------------ D1
    print("\nD1. MODEL SELECTION — which model is 'best' for each group?")
    print(f"{'Group':12s} {'best by raw F1':>18s} {'best by skill score':>21s} "
          f"{'agree?':>8s}")
    print("-" * 64)
    names = list(MACRO_F1)
    disagree_d1 = 0
    for i, g in enumerate(GROUPS):
        raw = [MACRO_F1[m][i] for m in names]
        sk = [skill_score(MACRO_F1[m][i], A[i]) for m in names]
        b_raw, b_sk = names[int(np.argmax(raw))], names[int(np.argmax(sk))]
        ok = b_raw == b_sk
        disagree_d1 += (not ok)
        print(f"{g:12s} {b_raw:>18s} {b_sk:>21s} {('yes' if ok else 'NO'):>8s}")
    print(f"\n  -> model selection changes in {disagree_d1}/{len(GROUPS)} groups.")
    print("     (Expected: within a group the ceiling is shared, so ranking is")
    print("      preserved. This is a NEGATIVE result and it is important: the")
    print("      confound does NOT invalidate model comparisons.)")

    # ------------------------------------------------------------ D2
    print("\nD2. REMEDIATION TARGETING — which group is worst served, per model?")
    print("    (this is where resources / extra data collection would be sent)")
    print(f"{'Model':18s} {'worst by raw F1':>17s} {'worst by skill':>16s} "
          f"{'agree?':>8s}")
    print("-" * 62)
    disagree_d2, flips = 0, []
    for m in names:
        raw = np.array(MACRO_F1[m])
        sk = skill_score(raw, A)
        w_raw, w_sk = GROUPS[int(np.argmin(raw))], GROUPS[int(np.argmin(sk))]
        ok = w_raw == w_sk
        disagree_d2 += (not ok)
        if not ok:
            flips.append((m, w_raw, w_sk))
        print(f"{m:18s} {w_raw:>17s} {w_sk:>16s} {('yes' if ok else 'NO'):>8s}")
    print(f"\n  -> remediation target changes in {disagree_d2}/{len(names)} models.")
    for m, a, b in flips:
        print(f"     {m}: raw F1 sends help to '{a}', skill score sends it to '{b}'")

    # ------------------------------------------------------------ magnitude
    print("\nRANK CORRELATION between the two criteria (per model, across groups)")
    print(f"{'Model':18s} {'Spearman rho':>14s}")
    print("-" * 34)
    rhos = []
    for m in names:
        raw = np.array(MACRO_F1[m]); sk = skill_score(raw, A)
        r1 = np.argsort(np.argsort(raw)); r2 = np.argsort(np.argsort(sk))
        rho = float(np.corrcoef(r1, r2)[0, 1])
        rhos.append(rho)
        print(f"{m:18s} {rho:14.3f}")
    print(f"{'MEAN':18s} {np.mean(rhos):14.3f}")

    print("\n" + "=" * 78)
    print("CONCLUSION")
    print("  * Model COMPARISON is robust to the confound (D1): within a group all")
    print("    models share the same ceiling, so their ranking is unchanged.")
    print(f"  * Group TARGETING is not (D2): the identified worst-served group")
    print(f"    changes for {disagree_d2}/{len(names)} models once the ceiling is removed.")
    print("    Under raw F1, the sparsest group looks worst almost by construction,")
    print("    so remediation is systematically directed at low-crime regions")
    print("    regardless of whether the model actually serves them poorly.")
    print("\n  This is the practical cost of the confound: it does not mislead us")
    print("  about which METHOD to use, but it does mislead us about WHERE the")
    print("  model is failing -- which is precisely the fairness question.")

    print("\nCAVEATS")
    print("  * Base rates are estimated from published crime totals (Poisson")
    print("    assumption); see literature_audit.py for the same caveat.")
    print("  * D2 assumes remediation targets the worst-served group; other")
    print("    policies (e.g. proportional allocation) would be affected")
    print("    differently, though the ordering issue is the same.")


if __name__ == "__main__":
    main()



In [ ]:
%%writefile preprocess_chicago.py
"""
Build the Chicago (2015) FedCrime dataset from the City of Chicago open-data
portal, producing ``data/chi_crime.csv`` with the schema the pipeline expects:

    date_occ, crime_type_id, neighborhood_id

The eight crime categories and their ids follow the paper's Chicago ordering:

    0 robbery   1 battery    2 deceptive_practice  3 burglary
    4 assault   5 theft      6 criminal_damage     7 narcotics

Two input modes
---------------
1. Direct API (default): pages the Socrata endpoint for the eight categories in
   2015. Requires network access on the machine you run this on.

       python scripts/preprocess_chicago.py --out data/chi_crime.csv

2. Local CSV: if you've downloaded the full "Crimes - 2001 to Present" CSV
   (columns include ``Date``, ``Primary Type``, ``Community Area``, ``Year``),
   point at it and skip the network:

       python scripts/preprocess_chicago.py --csv Crimes_-_2001_to_Present.csv \
           --out data/chi_crime.csv

Category counts should closely match the paper (theft ~57k, battery ~49k,
criminal damage ~29k, narcotics ~24k, assault ~17k, deceptive ~16k,
burglary ~13k, robbery ~10k).
"""
from __future__ import annotations

import argparse
import time

import pandas as pd

# paper Chicago primary_type -> crime_type_id
PRIMARY_TO_ID = {
    "ROBBERY": 0,
    "BATTERY": 1,
    "DECEPTIVE PRACTICE": 2,
    "BURGLARY": 3,
    "ASSAULT": 4,
    "THEFT": 5,
    "CRIMINAL DAMAGE": 6,
    "NARCOTICS": 7,
}
RESOURCE = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"


def from_api(year: int = 2015) -> pd.DataFrame:
    from urllib.parse import urlencode
    types = "','".join(PRIMARY_TO_ID)
    where = f"year={year} AND primary_type IN('{types}')"
    rows, offset, page = [], 0, 50000
    while True:
        # URL-encode the query so spaces/quotes in $where don't break the URL
        q = urlencode({"$select": "date,primary_type,community_area",
                       "$where": where, "$limit": page, "$offset": offset})
        url = f"{RESOURCE}?{q}"
        chunk = pd.read_json(url)
        if chunk.empty:
            break
        rows.append(chunk)
        offset += page
        print(f"  fetched {offset} rows...")
        time.sleep(0.5)
    return pd.concat(rows, ignore_index=True)


def from_csv(path: str, year: int = 2015) -> pd.DataFrame:
    df = pd.read_csv(path, usecols=["Date", "Primary Type",
                                    "Community Area", "Year"])
    df = df[df["Year"] == year]
    df = df[df["Primary Type"].isin(PRIMARY_TO_ID)]
    return df.rename(columns={"Date": "date", "Primary Type": "primary_type",
                              "Community Area": "community_area"})


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default=None, help="local Chicago crimes CSV")
    ap.add_argument("--year", type=int, default=2015)
    ap.add_argument("--out", default="data/chi_crime.csv")
    args = ap.parse_args()

    df = from_csv(args.csv, args.year) if args.csv else from_api(args.year)

    df = df.dropna(subset=["community_area", "primary_type", "date"])
    df["crime_type_id"] = df["primary_type"].str.upper().map(PRIMARY_TO_ID)
    df = df.dropna(subset=["crime_type_id"])
    df["crime_type_id"] = df["crime_type_id"].astype(int)
    # community areas 1..77 -> neighborhood_id 0..76
    df["neighborhood_id"] = (pd.to_numeric(df["community_area"], errors="coerce")
                             .astype("Int64") - 1)
    df = df.dropna(subset=["neighborhood_id"])
    df["date_occ"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")

    out = df[["date_occ", "crime_type_id", "neighborhood_id"]].copy()
    out.to_csv(args.out, index=False)
    print(f"\nWrote {len(out)} rows to {args.out}")
    print(f"regions: {out.neighborhood_id.nunique()}  "
          f"categories: {sorted(out.crime_type_id.unique())}")
    print(out.crime_type_id.value_counts().sort_index())


if __name__ == "__main__":
    main()



In [ ]:
%%writefile preprocess_cities.py
"""
Build FedCrime-format datasets for additional cities from open-data portals.

Produces ``data/<city>_crime.csv`` with the schema the pipeline expects:

    date_occ, crime_type_id, neighborhood_id

Each city is mapped onto EIGHT crime categories so results are comparable
across cities (as in the FedCrime paper's LA/Chicago setup).

Supported
---------
  nyc   New York City  — 77 police precincts   (Socrata qgea-i56i)
  sf    San Francisco  — ~41 analysis neighbourhoods (Socrata wg3w-h783)
  phl   Philadelphia   — 21 police districts   (Carto SQL API)

Usage
-----
    python scripts/preprocess_cities.py --city nyc --year 2019 --out data/nyc_crime.csv
    python scripts/preprocess_cities.py --city sf  --year 2019 --out data/sf_crime.csv
    python scripts/preprocess_cities.py --city phl --year 2019 --out data/phl_crime.csv

Requires network access on the machine you run this on.
"""
from __future__ import annotations

import argparse
import time
from urllib.parse import urlencode

import pandas as pd

# --------------------------------------------------------------------------- #
# City configurations: 8 categories each, mapped from the portal's own labels
# --------------------------------------------------------------------------- #
NYC = {
    "resource": "https://data.cityofnewyork.us/resource/qgea-i56i.json",
    "date": "cmplnt_fr_dt", "cat": "ofns_desc", "region": "addr_pct_cd",
    # 0 theft 1 assault 2 criminal-mischief 3 harassment
    # 4 robbery 5 burglary 6 narcotics 7 sex-crimes
    "map": {
        "PETIT LARCENY": 0, "GRAND LARCENY": 0,
        "ASSAULT 3 & RELATED OFFENSES": 1, "FELONY ASSAULT": 1,
        "CRIMINAL MISCHIEF & RELATED OF": 2,
        "HARRASSMENT 2": 3,
        "ROBBERY": 4,
        "BURGLARY": 5,
        "DANGEROUS DRUGS": 6,
        "SEX CRIMES": 7,
    },
    "names": ["theft", "assault", "criminal_mischief", "harassment",
              "robbery", "burglary", "narcotics", "sex_crimes"],
}

SF = {
    "resource": "https://data.sfgov.org/resource/wg3w-h783.json",
    "date": "incident_date", "cat": "incident_category",
    "region": "analysis_neighborhood",
    "map": {
        "Larceny Theft": 0,
        "Assault": 1,
        "Malicious Mischief": 2,
        "Other Miscellaneous": 3,
        "Robbery": 4,
        "Burglary": 5,
        "Drug Offense": 6,
        "Motor Vehicle Theft": 7,
    },
    "names": ["theft", "assault", "malicious_mischief", "other",
              "robbery", "burglary", "narcotics", "vehicle_theft"],
}

# SECOND DOMAIN (not crime): Chicago 311 service requests, 2019.
# Same 77 community areas as the Chicago crime data, so the spatial units and
# the Head/Mid/Tail construction are identical -- this isolates the DOMAIN as
# the only thing that changes.  Categories are chosen automatically as the eight
# most frequent request types (auto_top8), which avoids hard-coding a taxonomy.
CHI311 = {
    "resource": "https://data.cityofchicago.org/resource/v6vf-nfxy.json",
    "date": "created_date", "cat": "sr_type", "region": "community_area",
    "map": None,               # None -> take the 8 most frequent categories
    "names": None,
}

CITIES = {"nyc": NYC, "sf": SF, "chi311": CHI311}


def fetch_socrata(cfg, year, page=50000, max_rows=1_500_000):
    """Page a Socrata endpoint for one calendar year."""
    rows, offset = [], 0
    date, cat, region = cfg["date"], cfg["cat"], cfg["region"]
    where = (f"{date} >= '{year}-01-01T00:00:00.000' "
             f"AND {date} <= '{year}-12-31T23:59:59.000'")
    while offset < max_rows:
        q = urlencode({"$select": f"{date},{cat},{region}",
                       "$where": where, "$limit": page, "$offset": offset})
        chunk = pd.read_json(f"{cfg['resource']}?{q}")
        if chunk.empty:
            break
        rows.append(chunk)
        offset += page
        print(f"  fetched {offset} rows...")
        time.sleep(0.4)
    if not rows:
        raise SystemExit("No rows returned — check the year or the portal.")
    return pd.concat(rows, ignore_index=True)


def build(city: str, year: int, out: str, local_csv: str | None = None):
    cfg = CITIES[city]
    date, cat, region = cfg["date"], cfg["cat"], cfg["region"]

    df = pd.read_csv(local_csv) if local_csv else fetch_socrata(cfg, year)
    df = df.dropna(subset=[date, cat, region])

    mapping = cfg["map"]
    if mapping is None:                    # auto: eight most frequent categories
        top8 = df[cat].astype(str).str.strip().value_counts().head(8).index.tolist()
        mapping = {name: i for i, name in enumerate(top8)}
        cfg = dict(cfg, names=[t[:20] for t in top8])
        print("auto-selected categories:")
        for i, t in enumerate(top8):
            print(f"  {i} {t}")

    # map the portal's own labels onto the 8 shared categories
    df["crime_type_id"] = df[cat].astype(str).str.strip().map(mapping)
    df = df.dropna(subset=["crime_type_id"])
    df["crime_type_id"] = df["crime_type_id"].astype(int)

    # regions -> contiguous integer ids
    codes, uniques = pd.factorize(df[region].astype(str).str.strip())
    df["neighborhood_id"] = codes
    df = df[df["neighborhood_id"] >= 0]

    df["date_occ"] = pd.to_datetime(df[date]).dt.strftime("%Y-%m-%d")

    outdf = df[["date_occ", "crime_type_id", "neighborhood_id"]].copy()
    outdf.to_csv(out, index=False)

    print(f"\nWrote {len(outdf)} rows to {out}")
    print(f"regions: {outdf.neighborhood_id.nunique()} | "
          f"categories: {sorted(outdf.crime_type_id.unique())}")
    counts = outdf.crime_type_id.value_counts().sort_index()
    for i, n in counts.items():
        print(f"  {i} {cfg['names'][i]:20s} {n}")
    # sparsity preview (daily presence per region/category)
    days = pd.to_datetime(outdf.date_occ).nunique()
    R = outdf.neighborhood_id.nunique()
    cells = days * R * 8
    present = outdf.drop_duplicates(["date_occ", "neighborhood_id",
                                     "crime_type_id"]).shape[0]
    print(f"\napprox label sparsity: {100*(1-present/cells):.1f}% zeros "
          f"({days} days x {R} regions x 8 categories)")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--city", required=True, choices=list(CITIES))
    ap.add_argument("--year", type=int, default=2019)
    ap.add_argument("--out", required=True)
    ap.add_argument("--csv", default=None,
                    help="use a locally downloaded CSV instead of the API")
    args = ap.parse_args()
    build(args.city, args.year, args.out, args.csv)


if __name__ == "__main__":
    main()



## 2. Build all four cities + the second domain
(a few minutes; needs internet)


In [ ]:
!python preprocess_chicago.py --out data/chi_crime.csv
!python preprocess_cities.py --city nyc    --year 2019 --out data/nyc_crime.csv
!python preprocess_cities.py --city sf     --year 2019 --out data/sf_crime.csv
!python preprocess_cities.py --city chi311 --year 2019 --out data/chi311.csv


## 3. THEORY — propositions verified (fast, no GPU)
→ Paper Section: Theory


In [ ]:
!python theory.py 2>&1 | tee results/T1_theory.txt


In [ ]:
!python -m fairaudit.extensions 2>&1 | tee results/T2_extensions.txt


## 4. BASE-RATE ARTIFACT across 4 cities (fast, no model)
→ Paper Table 1


In [ ]:
!python baserate_analysis.py 2>&1 | tee results/T3_baserate.txt


## 5. LITERATURE AUDIT — 7 published models (fast)
→ Paper Tables 2–3


In [ ]:
!python reanalysis_published.py 2>&1 | tee results/T4_reanalysis.txt


In [ ]:
!python literature_audit.py    2>&1 | tee results/T5_litaudit.txt


## 6. DECISION IMPACT (fast)
→ Paper Table 4


In [ ]:
!python decision_impact.py 2>&1 | tee results/T6_decision.txt


## 7. METRIC TAXONOMY on real data — **the centrepiece** (GPU, ~20 min)
→ Paper Table 5. Confounded vs invariant metrics, per city.


In [ ]:
for city in ['la','chicago','nyc','sf']:
    print('\n'+'#'*72+f'\n### {city.upper()}\n'+'#'*72)
    !python robust_fair_gnn.py --city {city} --gnn gated --adathr --defense strust2 \
        --metrics --rounds {ROUNDS} --seeds 3 --save results/paper_results.jsonl \
        2>&1 | tee -a results/T7_metrics.txt


## 8. GNN COMPARISON — plain vs gated vs attention vs no-GNN (GPU, ~25 min)
→ Paper Table 6


In [ ]:
for city in ['la','chicago']:
    for g in ['plain','gated','attention']:
        print(f'\n===== {city} / {g} =====')
        !python robust_fair_gnn.py --city {city} --graph-compare --gnn {g} \
            --rounds {ROUNDS} 2>&1 | tee -a results/T8_gnn.txt


## 9. CLEAN BASELINE — all defenses, 10 seeds (GPU, ~40 min)
→ Paper Table 7


In [ ]:
!python robust_fair_gnn.py --city la --gnn gated --adathr --compare \
    --rounds {ROUNDS} --seeds {SEEDS} --save results/paper_results.jsonl \
    2>&1 | tee results/T9_clean.txt


## 10. ATTACKS — 3 attacks × all defenses × 10 seeds (GPU, ~90 min)
→ Paper Table 8. **The longest cell.** Re-run alone if the session drops.


In [ ]:
for atk in ['scale','labelflip','camouflage']:
    print('\n'+'#'*72+f'\n### ATTACK: {atk}\n'+'#'*72)
    !python robust_fair_gnn.py --city la --gnn gated --adathr --compare \
        --attack {atk} --attack-frac 0.17 --rounds {ROUNDS} --seeds {SEEDS} \
        --save results/paper_results.jsonl 2>&1 | tee -a results/T10_attacks.txt


## 11. CHICAGO attacks — generalisation (GPU, ~45 min)
→ Paper Table 9


In [ ]:
for atk in ['scale','camouflage']:
    print(f'\n### CHICAGO / {atk}')
    !python robust_fair_gnn.py --city chicago --gnn gated --adathr --compare \
        --attack {atk} --attack-frac 0.17 --rounds {ROUNDS} --seeds {SEEDS} \
        --save results/paper_results.jsonl 2>&1 | tee -a results/T11_chicago.txt


## 12. WEEK-AHEAD + UNCERTAINTY (GPU, ~15 min)
→ Paper Table 10


In [ ]:
for h in [1,3,7]:
    print(f'\n### horizon = {h} day(s)')
    !python robust_fair_gnn.py --city la --gnn gated --adathr --defense strust2 \
        --horizon {h} --rounds {ROUNDS} --seeds 3 --save results/paper_results.jsonl \
        2>&1 | tee -a results/T12_horizon.txt
print('\n### uncertainty (MC-dropout)')
!python robust_fair_gnn.py --city la --gnn gated --adathr --defense strust2 \
    --mc 10 --rounds {ROUNDS} 2>&1 | tee -a results/T12_horizon.txt


## 13. SECOND DOMAIN — Chicago 311 (not crime)
→ Paper Table 11


In [ ]:
import numpy as np, pandas as pd, sys; sys.path.insert(0,'.')
from fairaudit import audit
def load(path,start,end,C=8):
    df=pd.read_csv(path); df.date_occ=pd.to_datetime(df.date_occ,errors='coerce')
    df=df.dropna(subset=['date_occ'])
    days=pd.date_range(start,end,freq='D'); regs=sorted(df.neighborhood_id.unique())
    ri={r:i for i,r in enumerate(regs)}; di={d:i for i,d in enumerate(days)}
    M=np.zeros((len(days),len(regs),C),np.float32)
    for (d,r,k),_ in df.groupby([df.date_occ.dt.normalize(),'neighborhood_id','crime_type_id']).size().items():
        if d in di and r in ri and 0<=int(k)<C: M[di[d],ri[r],int(k)]=1
    return M
out=[]
for name,path,s,e in [('CRIME  (Chicago 2015)','data/chi_crime.csv','2015-01-01','2015-12-31'),
                      ('311    (Chicago 2019)','data/chi311.csv','2019-01-01','2019-12-31')]:
    M=load(path,s,e); tot=M.sum((0,2)); order=np.argsort(tot)[::-1]; R=len(tot)
    tag=np.array(['Mid']*R,dtype=object); tag[order[:round(R*.2)]]='Head'; tag[order[round(R*.5):]]='Tail'
    N=M.shape[0]; Y=M.reshape(N*R,-1); grp=np.tile(tag,N)
    S=np.full_like(Y,0.6)          # skill-free scores isolate the pure artifact
    rep=audit(Y,S,grp)
    out.append(f"\n######## {name} ########\n{rep}")
    print(out[-1])
open('results/T13_domain.txt','w').write("\n".join(out))


## 14. ✅ COLLECT EVERYTHING — run this last
Prints every result and zips them so you can download and send them.


In [ ]:
import os, json, glob
print('='*78); print('ALL RESULTS'); print('='*78)
for f in sorted(glob.glob('results/T*.txt')):
    print('\n'+'='*78); print('FILE:', f); print('='*78)
    print(open(f).read())
print('\n'+'='*78); print('MACHINE-READABLE ROWS (paper_results.jsonl)'); print('='*78)
if os.path.exists('results/paper_results.jsonl'):
    rows=[json.loads(l) for l in open('results/paper_results.jsonl')]
    print(f'{len(rows)} rows saved')
    for r in rows: print(json.dumps(r))
else:
    print('none saved - did the GPU cells run?')
!zip -qr paper_results.zip results/ && echo '\nZIPPED -> paper_results.zip (download from the file panel)'
